# D49 Auth Streamlit

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.10
제작 : 박광석 (모두의연구소)

# 인증과 화면 : Streamlit, 시스템 구조, API Key 인증, 파일 업로드

비동기 처리 노트에서 MNIST 추론 API에 비동기 처리, 에러 핸들링, 로깅을 더해 `app/main_final.py`를 완성했습니다. 오늘은 그 API를 비개발자도 쓸 수 있게 만들고 아무나 부를 수 없게 막습니다.

1\~5장에서는 Streamlit으로 대시보드를 만들어 API에 붙입니다. 6\~11장에서는 API Key 인증과 파일 업로드 안전장치를 갖춘 이미지 분류 API를 만듭니다. 12장에서는 사내에 올리기 전에 키를 코드 밖으로 옮기고 서버의 키 목록에 키 원문 대신 해시를 적습니다.

## 실행 환경

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 내용 |
|---|---|
| 기준 환경 | 코랩 Pro. GPU가 없어도 돕니다 |
| 앞 노트에서 가져올 것 | 비동기 처리 노트까지 만든 `app/` 폴더와 `models/mnist_state_dict.pth` (5.3절, 11장) |
| 버전 | 1.0절에서 고정해 설치합니다 |
| 띄우는 서버 | 5.3절 백엔드와 11장 API 서버는 8000번, 대시보드는 8501번, 12.3절 서버는 8490번 |
| 커널 재시작 | 같은 포트에 서버를 다시 띄우기 전 (5.3절, 11.2절, 11.5절). 코랩은 다시 시작한 뒤 **코랩에서 할 때** 셀부터 |
| 화면 | Streamlit 화면은 코랩에서 보이지 않아 `AppTest`로 확인합니다 (1.4절, 5.3절) |
| 오래 걸리는 셀 | 9.4절 1억 픽셀 그림 1\~2초. 나머지는 몇 초 이내 |

</div>

🖥 **코랩에서 할 때**  
코랩 런타임이 새로 뜨면 만든 파일이 사라집니다. REST API 노트에서 구글 드라이브에 만든 `model-serving-course` 폴더로 들어가서 시작합니다. 내 PC에서는 이 셀을 건너뛰고 `model-serving-course` 폴더에서 노트북을 엽니다.

세션을 다시 시작하면 작업 폴더가 `/content`로 돌아갑니다. 그때마다 이 셀부터 다시 실행합니다.

In [ ]:
# 코랩에서만 실행합니다. 내 PC에서는 건너뜁니다
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/model-serving-course"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("작업 폴더 :", os.getcwd())

🖥 **로컬 작업**
Streamlit 화면을 브라우저로 보는 일과 터미널 두 개를 띄우는 일(5.1절)은 여러분 PC에서 합니다.

## 1. Streamlit 소개: Python만으로 만드는 웹 UI

---

> **학습 목표**
> - Streamlit이 무엇이고, 왜 ML 엔지니어에게 유용한지 이해합니다.
> - Streamlit의 실행 방식(스크립트 재실행 모델)을 파악합니다.
> - 최소한의 코드로 Streamlit 앱을 실행해 봅니다.

### 1.0 사전 준비

> 이 섹션은 Streamlit 앱 파일을 직접 생성(`%%writefile`)하므로 지금까지 만든 파일에 의존하지 않습니다.
> 아래 셀에서 필요한 패키지와 폴더만 확인합니다.

In [1]:
!pip list | grep streamlit

'grep'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [2]:
import os
os.makedirs("frontend", exist_ok=True)

try:
    import streamlit
    print(f"✅ Streamlit: {streamlit.__version__}")
except ImportError:
    print("❌ Streamlit 미설치")
    print("  pip install streamlit")

✅ Streamlit: 1.38.0


이 노트가 확인한 버전으로 고정해 설치하고 깔린 버전을 찍어 둡니다(2026.10.07 확인). 이미 불러온 패키지의 버전이 바뀌었다면 런타임을 다시 시작한 뒤 이어 가십시오. 코랩은 다시 시작한 뒤 **코랩에서 할 때** 셀부터 실행합니다.

In [3]:
# 버전 고정 설치. torch와 torchvision은 코랩에 깔린 것을 그대로 씁니다
!pip install -q "streamlit>=1.64,<2" "fastapi>=0.141,<0.142" "uvicorn[standard]>=0.53,<0.54" \
                "pydantic>=2.13,<3" "python-multipart>=0.0.32,<0.1" "pillow>=12.3,<13" \
                "nest_asyncio>=1.6,<2" "requests>=2.32,<3"

# 실제로 깔린 버전을 찍어 둡니다. 결과가 달라졌을 때 가장 먼저 볼 기록입니다
from importlib.metadata import version

for name in ["streamlit", "fastapi", "uvicorn", "pydantic", "python-multipart",
             "pillow", "nest_asyncio", "requests", "torch", "torchvision"]:
    print(f"{name:17} {version(name)}")

streamlit         1.66.1
fastapi           0.141.1
uvicorn           0.53.0
pydantic          2.14.1
python-multipart  0.0.32
pillow            12.3.0
nest_asyncio      1.6.0
requests          2.34.2
torch             2.4.1
torchvision       0.19.1



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---

### 1.1 지금까지의 흐름 복습

지금까지 완성한 것을 정리합니다.

```
REST API 노트: 모델 학습 → 직렬화 → 파일 저장
FastAPI 기초 노트: FastAPI로 추론 API 구축 → Swagger UI에서 테스트
비동기 처리 노트: 비동기 처리, 에러 핸들링, 로깅으로 안정성 강화
```

API는 완성되었습니다. Swagger UI에서 테스트도 됩니다.
그런데 이 서비스를 **비개발자**에게 전달한다면 어떻게 될까요?

![image.png](lms-img:0)

사용자에게 필요한 것은 **이미지를 드래그 앤 드롭하면 결과가 나오는 화면**입니다.  
오늘은 그 화면을 만듭니다.

### 1.2 Streamlit이란 무엇입니까?

Streamlit은 **Python 스크립트만으로 웹 애플리케이션을 만드는 프레임워크**입니다.

```
일반적인 웹 UI 개발:
  HTML + CSS + JavaScript + React/Vue + 빌드 설정 + ...
  → 프론트엔드 전문 지식 필요
  → 수주일의 개발 기간

Streamlit:
  Python 스크립트 하나
  → 프론트엔드 지식 불필요
  → 수십 분이면 동작하는 UI 완성
```

ML 엔지니어에게 Streamlit이 특히 유용한 이유:

```
- 모델 데모를 빠르게 만들어 팀/고객에게 보여줄 수 있습니다.
- 내부 도구(데이터 시각화, 모니터링 대시보드)를 직접 만들 수 있습니다.
- Python만 알면 되므로, 프론트엔드 개발자 없이도 프로토타입이 가능합니다.
- 이 과정의 프로젝트(정형 데이터 프로젝트 부록, 챗봇 프로젝트 부록)에서 프론트엔드로 사용합니다.
```

### 1.3 Streamlit의 실행 모델: 스크립트 재실행

Streamlit의 동작 방식은 일반적인 웹 프레임워크와 다릅니다.
이것을 먼저 이해해야 이후 코드가 자연스럽게 읽힙니다.

```
[일반적인 웹 프레임워크]
  서버 시작 → 사용자 이벤트(클릭, 입력) → 해당 이벤트 핸들러만 실행

[Streamlit]
  사용자 이벤트(클릭, 입력) → 스크립트 전체를 위에서 아래로 다시 실행
```

즉, 사용자가 버튼을 누르거나 입력을 바꿀 때마다 **파이썬 스크립트가 처음부터 끝까지 다시 실행**됩니다.

```
app.py 실행 (1차)
─────────────────
  1행: st.title("...")        → 제목 렌더링
  2행: name = st.text_input() → 입력 위젯 렌더링 (값: "")
  3행: st.write(f"안녕, {name}") → "안녕, " 출력

사용자가 이름을 입력 → "홍길동"

app.py 실행 (2차, 전체 재실행)
─────────────────
  1행: st.title("...")        → 제목 렌더링
  2행: name = st.text_input() → 입력 위젯 렌더링 (값: "홍길동")
  3행: st.write(f"안녕, {name}") → "안녕, 홍길동" 출력
```

> 이 동작 방식 때문에 **무거운 작업(모델 로드 등)을 매번 재실행하면 느려집니다.**
> 이 문제를 해결하는 `st.cache_resource`는 섹션 2에서 다룹니다.

### 1.4 실습: 첫 번째 Streamlit 앱

> **주의** : **Streamlit은 주피터 노트북 안에서 직접 실행할 수 없습니다.**
>
> Streamlit은 독립적인 웹 서버로 동작하므로, `.py` 파일로 작성한 뒤
> 터미널에서 `streamlit run` 명령어로 실행합니다.
>
> 주피터 노트북에서는 `%%writefile`로 파일을 생성하고,
> `!streamlit run`으로 실행하는 방식을 사용합니다.

In [4]:
%%writefile frontend/app_hello.py
"""
Day 4 - 첫 번째 Streamlit 앱
"""
import streamlit as st

# 페이지 설정
st.set_page_config(
    page_title="My First Streamlit App",
    page_icon="🤖",
    layout="centered",
)

# 제목
st.title("🤖 나의 첫 Streamlit 앱")
st.write("Python 코드만으로 이 화면이 만들어졌습니다.")

# 구분선
st.divider()

# 텍스트 입력
name = st.text_input("이름을 입력하세요:", placeholder="홍길동")

# 조건부 출력
if name:
    st.success(f"안녕하세요, {name}님! 환영합니다. 🎉")
else:
    st.info("위에 이름을 입력해 보세요.")

# 버튼
if st.button("날짜 확인"):
    from datetime import datetime
    now = datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분")
    st.write(f"현재 시각: {now}")

Writing frontend/app_hello.py


#### 실행 방법

```bash
# 터미널에서 실행 (별도 터미널을 열거나, 노트북에서 !로 실행)
# 주의: !로 실행하면 노트북 셀이 블로킹되므로, 별도 터미널 권장

# 로컬 환경:
streamlit run frontend/app_hello.py --server.port 8501

# Colab 환경:
# !streamlit run frontend/app_hello.py --server.port 8501 &
# (백그라운드 실행 후 ngrok 또는 localtunnel로 외부 접속)
```

> 실행하면 브라우저가 자동으로 열리며 `http://localhost:8501`에 앱이 표시됩니다.
> 이름을 입력하고, 버튼을 눌러보세요.
> **Python 코드 17줄로 인터랙티브한 웹 페이지**가 만들어진 것입니다.

![image.png](lms-img:0)

```python
# 노트북에서 백그라운드로 실행하는 방법 (선택)
import subprocess, time

proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_hello.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ Streamlit 앱 실행 중: http://localhost:8501")
print("   브라우저에서 위 주소로 접속하세요.")

# Colab:
# from google.colab import output
# output.serve_kernel_port_as_iframe(8501)
```

🖥 **로컬 작업**
`streamlit run`으로 띄운 화면은 브라우저에서 봅니다. 코랩에서 띄운 Streamlit은 구글의 기계에서 돌기 때문에 여러분 브라우저에 `localhost:8501`을 쳐도 닿지 않습니다. 화면은 여러분 PC에서 확인하십시오.

🔎 **더 알아보기**
실행 방법의 주석에 적힌 ngrok과 localtunnel은 코랩 안의 화면을 **공개 인터넷 주소**로 내보내는 도구입니다. 사내 데이터를 띄우는 화면이라면 연습이라도 이 길은 쓰지 않습니다. 코랩에서는 Streamlit이 주는 `AppTest`로 브라우저 없이 스크립트를 실제로 돌려 봅니다.

In [1]:
# AppTest : 브라우저 없이 Streamlit 스크립트를 실제로 돌려 봅니다
import os

from streamlit.testing.v1 import AppTest

# 상대 경로는 이 셀이 저장된 자리를 기준으로 찾으므로 절대 경로로 넘깁니다
at = AppTest.from_file(os.path.abspath("frontend/app_hello.py")).run()
print("제목      :", at.title[0].value)
print("안내      :", [e.value for e in at.info])

at.text_input[0].set_value("홍길동").run()      # 이름 칸에 글자를 넣은 것과 같습니다
print("입력 뒤   :", [e.value for e in at.success])
print("남은 안내 :", [e.value for e in at.info])

2026-10-30 14:29:10.624 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


제목      : 🤖 나의 첫 Streamlit 앱
안내      : ['위에 이름을 입력해 보세요.']
입력 뒤   : ['안녕하세요, 홍길동님! 환영합니다. 🎉']
남은 안내 : []


이름을 넣기 전에는 `st.info`가, 넣은 뒤에는 `st.success`가 그려졌습니다. 입력이 바뀌자 1.3절의 설명대로 스크립트가 처음부터 다시 돌았고 `if name:` 분기가 갈렸습니다. `AppTest`를 돌릴 때마다 찍히는 `missing ScriptRunContext` 경고는 화면 없이 돌릴 때 나오는 안내라 무시해도 됩니다.

### 1.5 코드 해부: 한 줄씩 이해하기

```python
import streamlit as st
```

Streamlit의 모든 기능은 `st` 모듈을 통해 접근합니다.

```python
st.set_page_config(page_title="...", page_icon="🤖", layout="centered")
```

페이지 설정입니다. 브라우저 탭 제목, 아이콘, 레이아웃을 지정합니다.
**반드시 스크립트의 가장 첫 번째 Streamlit 호출**이어야 합니다.

```python
st.title("🤖 나의 첫 Streamlit 앱")    # 큰 제목 (H1)
st.write("Python 코드만으로...")         # 범용 출력
st.divider()                            # 수평 구분선
```

텍스트 출력 함수들입니다. `st.write()`는 입력 타입에 따라 자동으로 렌더링합니다.

```python
name = st.text_input("이름을 입력하세요:", placeholder="홍길동")
```

입력 위젯입니다. 사용자가 값을 입력하면 **스크립트가 재실행**되면서 `name` 변수에 새 값이 들어옵니다.

```python
st.success("...")    # 초록색 성공 메시지
st.info("...")       # 파란색 정보 메시지
st.warning("...")    # 노란색 경고 메시지
st.error("...")      # 빨간색 에러 메시지
```

상태별 메시지 박스입니다. API 응답 결과를 사용자에게 보여줄 때 유용합니다.

---

### 체크포인트

1. Streamlit의 스크립트 재실행 모델이란 무엇입니까?
2. `st.text_input()`에 값을 입력하면 내부적으로 어떤 일이 일어납니까?
3. `st.set_page_config()`를 스크립트 중간에 호출하면 어떻게 됩니까?

---

> **다음 섹션에서는** Streamlit의 핵심 컨셉(위젯, 레이아웃, 캐시)을 간략히 다룹니다.

## 2. Streamlit 핵심 컨셉

---

> **학습 목표**
> - Streamlit의 주요 위젯과 레이아웃 요소를 파악합니다.
> - `st.cache_resource`의 역할을 이해합니다.
> - 섹션 5 대시보드 실습에 필요한 기초를 갖춥니다.

---

### 2.1 자주 사용하는 위젯

Streamlit은 다양한 입력 위젯을 제공합니다.
아래는 이 과정 프로젝트에서 실제로 사용하는 것들입니다.

```python
# 파일 업로드 : 이미지 추론의 핵심 입력
uploaded = st.file_uploader("이미지 업로드:", type=["png", "jpg"])

# 버튼 : 추론 실행 트리거
if st.button("추론 실행"):
    ...

# 체크박스 : 옵션 토글
show_probs = st.checkbox("전체 확률 표시", value=True)

# 셀렉트박스 : 선택지 제공
method = st.radio("입력 방식:", ["파일 업로드", "샘플 이미지"], horizontal=True)

# 스피너 : 추론 중 로딩 표시
with st.spinner("모델 추론 중..."):
    result = call_api(...)

# 상태 메시지
st.success("✅ 추론 완료!")
st.error("❌ 서버에 연결할 수 없습니다.")
st.info("이미지를 업로드하세요.")
```

> 위젯은 외울 필요 없습니다. 섹션 5에서 대시보드를 만들면서 자연스럽게 익히게 됩니다.
> 전체 위젯 목록은 Streamlit 공식 문서(https://docs.streamlit.io)에서 확인할 수 있습니다.

---

### 2.2 레이아웃 기초

프로젝트에서 사용하는 레이아웃 요소는 세 가지입니다.

```python
# 1. 사이드바: 설정, 옵션 배치
with st.sidebar:
    st.header("⚙️ 설정")
    show_probs = st.checkbox("확률 표시")

# 2. 컬럼: 입력과 결과를 나란히 배치
col1, col2 = st.columns(2)
with col1:
    st.subheader("입력")
with col2:
    st.subheader("결과")

# 3. 전체 너비 사용
st.set_page_config(layout="wide")
```

---

### 2.3 st.cache_resource

섹션 1에서 Streamlit은 이벤트마다 스크립트를 재실행한다고 했습니다.
매번 재실행할 때마다 객체를 다시 생성하면 느려지므로, **한 번만 생성하고 재사용**하는 캐시가 필요합니다.

In [2]:
@st.cache_resource
def get_client():
    """앱이 재시작되기 전까지 한 번만 생성됩니다."""
    return APIClient("http://localhost:8000")

client = get_client()   # 스크립트가 몇 번 재실행되든 같은 객체

NameError: name 'st' is not defined

🔎 **더 알아보기**
이 셀은 대시보드 파일 안에서 쓰는 모양을 보여 주는 코드라 노트북에서 실행하면 `NameError: name 'st' is not defined`가 납니다. 코랩의 모두 실행은 여기서 멈추므로 이 셀은 건너뛰십시오.

> 이 과정에서는 API 클라이언트를 캐시하는 용도로만 사용합니다.
> 모델은 FastAPI 서버가 관리하므로, Streamlit에서 모델을 로드할 일은 없습니다.

---

### 체크포인트

1. `st.file_uploader()`로 업로드된 파일의 바이트 데이터는 어떻게 얻습니까?
2. Streamlit에서 `@st.cache_resource`를 사용하는 이유는 무엇입니까?

---

> **다음 섹션에서는** 프론트엔드와 백엔드를 왜 분리해야 하는지 살펴봅니다.

## 3. System Architecture: Frontend와 Backend의 역할 분리

---

> **학습 목표**
> - 프론트엔드와 백엔드를 분리하는 이유를 설명할 수 있습니다.
> - 모놀리식과 분리 아키텍처의 차이를 이해합니다.
> - 이 과정에서 만드는 서비스의 전체 아키텍처를 그릴 수 있습니다.

---

### 3.1 두 가지 접근법

Streamlit에서 모델 추론을 수행하는 방법은 두 가지가 있습니다.

![image.png](lms-img:0)

이 과정에서는 **방법 B(분리 아키텍처)**를 사용합니다.

---

### 3.2 왜 분리해야 합니까?

모놀리식이 더 간단한데 굳이 분리하는 핵심 이유 두 가지를 살펴보겠습니다.

#### 이유 1: 독립적인 개발·배포·확장

```
분리 아키텍처에서는:
  - 모델을 업데이트할 때 → FastAPI 서버만 재배포
  - UI를 변경할 때 → Streamlit 앱만 재배포
  - 사용자가 늘어서 추론 부하가 높아지면 → FastAPI만 추가 배포 (Scale-out)

모놀리식에서는:
  - 모든 변경이 앱 전체 재배포를 요구합니다.
  - 추론 부하를 처리하려면 UI까지 같이 복제해야 합니다.
```

#### 이유 2: 다양한 클라이언트 지원

```
FastAPI 서버가 독립적으로 존재하면:

  Streamlit 앱  ──▶
  모바일 앱     ──▶  FastAPI 서버  ──▶  모델 추론
  다른 서버     ──▶
  curl / 스크립트──▶

→ 하나의 API로 여러 클라이언트를 지원할 수 있습니다.
→ Streamlit은 여러 클라이언트 중 하나일 뿐입니다.
```

---

### 3.3 이 과정의 전체 아키텍처

추후 진행할 과정까지, 이 모듈에서 구축하는 시스템의 전체 구조입니다.

![image.png](lms-img:0)

각 레이어의 역할을 정리합니다:

```
Streamlit (Frontend) : Port 8501
  역할: 사용자와의 상호작용
  담당: UI 렌더링, 파일 업로드 수신, API 호출, 결과 시각화
  기술: Streamlit

FastAPI (Backend) : Port 8000
  역할: 비즈니스 로직 처리
  담당: 입력 검증, 전처리, 모델 추론, 후처리, 에러 핸들링, 로깅
  기술: FastAPI, Pydantic, asyncio, PyTorch

모델 파일 (Storage)
  역할: 학습된 모델 가중치 저장
  담당: 직렬화된 모델 파일 (.pth, .onnx 등)
```

---

### 3.4 통신 흐름

사용자가 이미지를 업로드했을 때의 전체 흐름을 추적합니다.

```
사용자: 이미지 드래그 앤 드롭

  ① Streamlit: 파일 수신
  ② Streamlit: 이미지 → Base64 변환
  ③ Streamlit: POST http://localhost:8000/predict/image
               {"image_base64": "iVBOR...", "return_probabilities": true}
  
  ④ FastAPI: Pydantic으로 입력 검증
  ⑤ FastAPI: Base64 → PIL Image → 전처리 → 텐서
  ⑥ FastAPI: run_in_executor → model(tensor)
  ⑦ FastAPI: 후처리 → JSON 응답
               {"success": true, "predicted_class": "7", "confidence": 0.98, ...}
  
  ⑧ Streamlit: 응답 수신
  ⑨ Streamlit: 결과를 st.metric, st.bar_chart 등으로 시각화

사용자: 결과 확인
```

> 이 흐름에서 Streamlit은 **모델에 대해 아무것도 알지 못합니다.**
> PyTorch가 설치되어 있지 않아도 됩니다.
> HTTP 요청을 보내고, JSON 응답을 받아서 화면에 그릴 뿐입니다.

---

### 체크포인트

1. 모놀리식과 분리 아키텍처의 핵심 차이를 한 문장으로 설명하세요.
2. 모델을 업데이트할 때, 분리 아키텍처에서는 어떤 서버만 재배포하면 됩니까?
3. Streamlit 앱에 PyTorch가 설치되어 있지 않아도 되는 이유는 무엇입니까?

---

> **다음 섹션에서는** Streamlit에서 FastAPI를 호출하는 코드를 작성합니다.
> `requests`와 `httpx`로 HTTP 요청을 보내는 방법을 실습합니다.

## 4. Streamlit에서 FastAPI 호출하기

---

> **학습 목표**
> - `requests` 라이브러리로 Streamlit에서 FastAPI를 호출할 수 있습니다.
> - 이미지를 Base64로 변환하여 API에 전송할 수 있습니다.
> - API 호출 실패 시 사용자에게 적절한 피드백을 제공할 수 있습니다.

---

### 4.1 기본 패턴

Streamlit에서 FastAPI를 호출하는 코드는 단순합니다.

```python
import requests

# GET : 서버 상태 확인
response = requests.get("http://localhost:8000/health")
health = response.json()

# POST : 추론 요청 (Base64 이미지)
import base64

image_base64 = base64.b64encode(image_bytes).decode("utf-8")
response = requests.post(
    "http://localhost:8000/predict/image",
    json={"image_base64": image_base64, "return_probabilities": True},
)
result = response.json()
```

> REST API 노트에서 `requests.get()`으로 외부 API를 호출한 것과 동일합니다.
> 차이점은 호출 대상이 **우리가 만든 FastAPI 서버**라는 것뿐입니다.

---

### 4.2 에러 처리

API 호출은 실패할 수 있습니다. `try/except`로 감싸서 사용자에게 안내합니다.

```python
import streamlit as st
import requests

def call_api(url, json_data=None, method="post"):
    """API를 호출하고, 실패 시 st.error()로 안내합니다."""
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버가 실행 중인지 확인하세요.")
        return None
    except requests.exceptions.Timeout:
        st.warning("⏱️ **응답 시간 초과.** 잠시 후 다시 시도하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None
```

> 핵심 원칙: **성공 시 dict, 실패 시 None을 반환.**
> 호출 측에서 `if result:`로 간단하게 분기할 수 있습니다.

---

### 체크포인트

1. 이미지를 API에 전송할 때 Base64로 인코딩하는 이유는 무엇입니까?
2. `response.raise_for_status()`는 어떤 역할을 합니까?

---

> **다음 섹션에서는** 오늘 배운 모든 내용을 종합하여
> MNIST 추론 대시보드를 처음부터 끝까지 구현합니다.

## 5. 실습: MNIST 추론 대시보드 만들기

---

![image.png](lms-img:0)

> **실습 목표**
> - MNIST 추론 대시보드를 처음부터 끝까지 구현합니다.
> - 이미지 업로드 → API 호출 → 결과 시각화의 전체 흐름을 완성합니다.
> - 지금까지 배운 모든 내용이 하나의 서비스로 동작하는 것을 확인합니다.

---

### 5.1 전체 구성

```
실행해야 하는 서버: 2개

터미널 1 (백엔드):  uvicorn app.main_final:app --port 8000
터미널 2 (프론트엔드): streamlit run frontend/app_dashboard.py --server.port 8501
```

---

### 5.2 대시보드 코드 작성

백엔드는 FastAPI로, 프론트엔드는 Streamlit으로 구동되는 MNIST 추론 대시보드를 구성하는 코드를 작성합니다.

In [3]:
%%writefile frontend/app_dashboard.py
"""
Day 4 - MNIST 추론 대시보드
FastAPI 백엔드와 연동하는 Streamlit 프론트엔드
"""
import streamlit as st
import requests
import base64
import io
from PIL import Image


# ===== 페이지 설정 =====
st.set_page_config(
    page_title="MNIST 숫자 인식",
    page_icon="🔢",
    layout="wide",
)


# ===== API 호출 함수 =====
API_BASE = "http://localhost:8000"

def call_api(url, json_data=None, method="post"):
    """API를 호출하고, 실패 시 에러 메시지를 표시합니다."""
    try:
        if method == "get":
            resp = requests.get(url, timeout=10)
        else:
            resp = requests.post(url, json=json_data, timeout=30)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 **서버에 연결할 수 없습니다.** FastAPI 서버가 실행 중인지 확인하세요.")
        return None
    except requests.exceptions.Timeout:
        st.warning("⏱️ **응답 시간 초과.** 잠시 후 다시 시도하세요.")
        return None
    except requests.exceptions.HTTPError as e:
        st.error(f"❌ **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f"❌ **오류:** {type(e).__name__}")
        return None


# ===== 사이드바 =====
with st.sidebar:
    st.header("⚙️ 설정")

    # 서버 상태 표시
    health = call_api(f"{API_BASE}/health", method="get")
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        server_ok = True
    else:
        st.error("🔴 서버 연결 실패")
        server_ok = False

    st.divider()

    # 옵션
    show_probabilities = st.checkbox("전체 확률 표시", value=True)
    show_preprocessed = st.checkbox("전처리된 이미지 표시", value=True)

    st.divider()
    st.caption("MNIST Prediction Dashboard v1.0")


# ===== 메인 영역 =====
st.title("🔢 MNIST 숫자 인식")
st.write("손글씨 숫자 이미지를 업로드하면 0~9 중 어떤 숫자인지 예측합니다.")

col_input, col_result = st.columns(2)

# ----- 입력 영역 -----
with col_input:
    st.subheader("📤 이미지 입력")

    input_method = st.radio(
        "입력 방식:", ["파일 업로드", "샘플 이미지 사용"], horizontal=True,
    )

    image_bytes = None

    if input_method == "파일 업로드":
        uploaded = st.file_uploader(
            "이미지를 업로드하세요:",
            type=["png", "jpg", "jpeg"],
            help="28x28 그레이스케일 권장. 다른 크기도 자동 변환됩니다.",
        )
        if uploaded:
            image_bytes = uploaded.getvalue()
            st.image(uploaded, caption="업로드된 이미지", width=200)

    else:
        st.info("샘플 이미지를 사용하려면 아래 버튼을 누르세요.")
        sample_idx = st.number_input("샘플 번호 (0~99):", min_value=0, max_value=99, value=0)

        if st.button("샘플 이미지 로드"):
            try:
                from torchvision import datasets
                test_dataset = datasets.MNIST(root="data", train=False, download=True)
                sample_image, sample_label = test_dataset[sample_idx]

                buffer = io.BytesIO()
                sample_image.save(buffer, format="PNG")
                image_bytes = buffer.getvalue()

                st.image(sample_image, caption=f"샘플 #{sample_idx} (정답: {sample_label})", width=200)
                st.session_state["sample_label"] = sample_label
            except Exception as e:
                st.error(f"샘플 로드 실패: {e}")

    # 전처리된 이미지 미리보기
    if image_bytes and show_preprocessed:
        st.caption("전처리된 이미지 (28x28 그레이스케일):")
        img = Image.open(io.BytesIO(image_bytes)).convert("L").resize((28, 28))
        st.image(img, width=150)


# ----- 결과 영역 -----
with col_result:
    st.subheader("📊 추론 결과")

    if image_bytes is None:
        st.info("👈 왼쪽에서 이미지를 업로드하거나 샘플을 선택하세요.")

    elif not server_ok:
        st.error("서버에 연결할 수 없습니다. 사이드바의 서버 상태를 확인하세요.")

    else:
        if st.button("🚀 추론 실행", type="primary", use_container_width=True):
            with st.spinner("모델 추론 중..."):
                # Base64 인코딩 → API 호출
                image_base64 = base64.b64encode(image_bytes).decode("utf-8")
                result = call_api(
                    f"{API_BASE}/predict/image",
                    json_data={
                        "image_base64": image_base64,
                        "return_probabilities": show_probabilities,
                    },
                )

            if result:
                st.session_state["last_result"] = result

        # 결과 표시
        if "last_result" in st.session_state:
            result = st.session_state["last_result"]

            # 메트릭
            m1, m2 = st.columns(2)
            with m1:
                st.metric(label="예측 결과", value=result["predicted_class"])
            with m2:
                st.metric(label="확신도", value=f"{result['confidence']:.1%}")

            # 확률 분포
            if result.get("probabilities"):
                st.subheader("📊 클래스별 확률 분포")
                probs = result["probabilities"]
                for cls in sorted(probs.keys(), key=lambda x: int(x)):
                    prob = probs[cls]
                    c1, c2 = st.columns([1, 5])
                    with c1:
                        is_pred = cls == result["predicted_class"]
                        st.write(f"**{'👉 ' if is_pred else ''}{cls}**")
                    with c2:
                        st.progress(float(prob), text=f"{prob:.2%}")

            # 샘플 이미지인 경우 정답 비교
            if "sample_label" in st.session_state:
                label = st.session_state["sample_label"]
                if result["predicted_class"] == str(label):
                    st.success(f"✅ 정답! (정답: {label})")
                else:
                    st.error(f"❌ 오답 (정답: {label}, 예측: {result['predicted_class']})")

Writing frontend/app_dashboard.py


🔎 **더 알아보기**
`st.button(..., use_container_width=True)`는 이 노트의 버전(Streamlit 1.65)에서 경고 없이 돌지만 공식 문서에는 폐기 예정으로 적혀 있습니다. 새로 쓰는 코드에서는 같은 뜻의 `width="stretch"`를 씁니다.

---

### 5.3 실행 및 테스트

#### Step 1: 백엔드 서버 실행

🖥 코랩에서 세션을 다시 시작했다면 **코랩에서 할 때** 셀부터 실행한 뒤 서버를 띄웁니다. 건너뛰면 작업 폴더가 `/content`라서 서버 스레드가 `ModuleNotFoundError: No module named 'app'`로 멈추는데도 서버 셀은 성공 문구를 찍습니다.

In [ ]:
# ⚠️ 이전 Day/섹션에서 서버를 실행했다면 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_backend():
    uvicorn.run("app.main_final:app", host="0.0.0.0", port=8000)

backend_thread = threading.Thread(target=run_backend, daemon=True)
backend_thread.start()
time.sleep(3)
print("✅ 백엔드 서버: http://localhost:8000")

✅ 백엔드 서버: http://localhost:8000


INFO:     Started server process [21876]
INFO:     Waiting for application startup.


2026-10-30 14:42:52 INFO     [ml_api] 모델 로드 중: models/mnist_state_dict.pth
2026-10-30 14:42:52 INFO     [ml_api] 모델 로드 완료


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-10-30 14:43:51 INFO     [ml_api] GET /health → 200 (0.0s)
INFO:     127.0.0.1:9255 - "GET /health HTTP/1.1" 200 OK
2026-10-30 14:50:03 WARNING  [ml_api] GET / → 404 (0.0s)
INFO:     199.165.159.17:40100 - "GET / HTTP/1.1" 404 Not Found
2026-10-30 14:50:39 WARNING  [ml_api] GET /favicon.ico → 404 (0.001s)
INFO:     199.165.159.23:50767 - "GET /favicon.ico HTTP/1.1" 404 Not Found
2026-10-30 14:51:04 WARNING  [ml_api] GET / → 404 (0.0s)
INFO:     199.165.159.17:24323 - "GET /?format=json HTTP/1.1" 404 Not Found


#### Step 2: 프론트엔드 실행

![image.png](lms-img:0)

🔎 **더 알아보기**
그림의 `Network URL` 줄은 같은 망의 다른 컴퓨터가 이 화면에 들어오는 주소입니다. Step 1의 백엔드도 `host="0.0.0.0"`으로 떠서 같은 망에 열려 있습니다. 혼자 테스트할 때는 `--server.address 127.0.0.1`과 `host="127.0.0.1"`로 이 컴퓨터에서만 닿게 하고 인증(7장)을 건 뒤에 엽니다.

그림의 `Summary` 안내대로 Streamlit은 기본 설정에서 사용 통계를 미국의 서버로 보냅니다. 사내 PC나 폐쇄망에서는 안내에 적힌 대로 `.streamlit/config.toml`의 `[browser]`에 `gatherUsageStats = false`를 적어 끕니다.

In [ ]:
# 별도 터미널에서 실행합니다
!cd model-serving-course
!streamlit run frontend/app_dashboard.py --server.port 8501

In [ ]:
# 또는 노트북에서 백그라운드 실행
import subprocess
proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_dashboard.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print("✅ 프론트엔드: http://localhost:8501")

# Colab: from google.colab import output
# output.serve_kernel_port_as_iframe(8501)

#### Step 3: 테스트 시나리오

```
브라우저에서 http://localhost:8501 에 접속합니다.

[테스트 1] 기본 흐름
  1. 사이드바에서 서버 상태가 초록인지 확인
  2. "샘플 이미지 사용" → 샘플 번호 0 → "샘플 이미지 로드"
  3. "추론 실행" 클릭
  4. 예측 숫자, 확신도, 확률 분포가 표시되는지 확인

[테스트 2] 파일 업로드
  1. "파일 업로드" 선택 → 아무 이미지 드래그 앤 드롭
  2. "추론 실행" → 결과 확인

[테스트 3] 에러 상황
  1. 백엔드 서버를 종료
  2. 사이드바 상태가 빨강으로 변하는지 확인
```

이미지 입력 후

![image.png](lms-img:0)

추론실행 버튼을 누르면 결과가 출력됩니다.   
우리의 모델이 3 epoch만 학습이 되어서 일을 잘 못하지만, streamlit으로 만든 front 페이지는 훌륭하게 동작합니다!

![image.png](lms-img:0)

샘플 데이터로 테스트 해보실 수도 있습니다.

![image.png](lms-img:0)

🖥 **로컬 작업**
Step 2의 `!streamlit run` 셀은 서버가 끝나지 않으므로 셀도 끝나지 않습니다. 코랩에서는 중지 버튼으로 멈추고 백그라운드 실행 셀로 띄우십시오. 화면이 보이지 않으므로 Step 3의 시나리오는 1.4절에서 쓴 `AppTest`로 대신 확인합니다.

코랩에서 이 셀의 `!cd model-serving-course` 줄은 늘 `No such file or directory`를 찍습니다. 작업 폴더는 **코랩에서 할 때** 셀이 이미 정했으므로 이 문구는 무시해도 됩니다.

In [2]:
# 5장 대시보드를 브라우저 없이 돌립니다. Step 1의 백엔드가 떠 있어야 합니다
import io
import os

from PIL import Image
from streamlit.testing.v1 import AppTest

at = AppTest.from_file(os.path.abspath("frontend/app_dashboard.py"), default_timeout=30).run()
print("사이드바 :", [e.value for e in list(at.sidebar.success) + list(at.sidebar.error)])

buf = io.BytesIO()
Image.new("L", (28, 28), 0).save(buf, format="PNG")      # 까만 28x28 그림 한 장
at.file_uploader[0].set_value(("black.png", buf.getvalue(), "image/png")).run()
buttons = [b for b in at.button if "추론" in b.label]
if not buttons:
    print("추론 실행 버튼이 없습니다. 사이드바 상태를 보고 Step 1의 백엔드부터 띄우십시오")
else:
    buttons[0].click().run()
    print("[테스트 2] 파일 업로드 뒤 :", [(m.label, m.value) for m in at.metric])

2026-10-30 15:44:38.650 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-10-30 15:44:41 INFO     [ml_api] GET /health → 200 (0.0s)
INFO:     127.0.0.1:2587 - "GET /health HTTP/1.1" 200 OK


2026-10-30 15:44:41.159 127.0.0.1:2587 - "GET /health HTTP/1.1" 200


사이드바 : ['서버 연결됨']
2026-10-30 15:44:43 INFO     [ml_api] GET /health → 200 (0.0s)
INFO:     127.0.0.1:2591 - "GET /health HTTP/1.1" 200 OK


2026-10-30 15:44:43.275 127.0.0.1:2591 - "GET /health HTTP/1.1" 200


2026-10-30 15:44:45 INFO     [ml_api] GET /health → 200 (0.0s)
INFO:     127.0.0.1:2595 - "GET /health HTTP/1.1" 200 OK


2026-10-30 15:44:45.423 127.0.0.1:2595 - "GET /health HTTP/1.1" 200


2026-10-30 15:44:47 INFO     [ml_api] POST /predict/image → 200 (0.072s)
INFO:     127.0.0.1:2597 - "POST /predict/image HTTP/1.1" 200 OK


2026-10-30 15:44:47.549 127.0.0.1:2597 - "POST /predict/image HTTP/1.1" 200


[테스트 2] 파일 업로드 뒤 : [('예측 결과', '7'), ('확신도', '11.8%')]


🔎 **더 알아보기**
사내에 올리면 대시보드와 백엔드는 다른 기계에 섭니다. `API_BASE`를 코드에 적어 두면 주소가 바뀔 때마다 코드를 고쳐야 하므로 주소 한 줄만 환경 변수 `D49_API_BASE`에서 읽게 바꾼 사본으로 [테스트 3]을 합니다. 백엔드를 끄는 대신 아무도 없는 주소를 넣습니다.

In [3]:
# 주소 한 줄만 환경 변수에서 읽게 바꾼 사본을 만듭니다. 원본 파일은 그대로 둡니다
src = open("frontend/app_dashboard.py", encoding="utf-8").read()
src = src.replace('API_BASE = "http://localhost:8000"',
                  'import os\nAPI_BASE = os.environ.get("D49_API_BASE", "http://localhost:8000")')
open("frontend/app_dashboard_env.py", "w", encoding="utf-8").write(src)

for base in ["http://localhost:8499", "http://localhost:8000"]:   # 아무도 없는 주소, Step 1의 백엔드
    os.environ["D49_API_BASE"] = base
    at = AppTest.from_file(os.path.abspath("frontend/app_dashboard_env.py"), default_timeout=30).run()
    print(f"[테스트 3] {base} ->", [e.value for e in list(at.sidebar.success) + list(at.sidebar.error)])

2026-10-30 15:44:52.707 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-30 15:44:57.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


[테스트 3] http://localhost:8499 -> ['**서버에 연결할 수 없습니다.** FastAPI 서버가 실행 중인지 확인하세요.', '서버 연결 실패']
2026-10-30 15:44:59 INFO     [ml_api] GET /health → 200 (0.0s)
INFO:     127.0.0.1:11965 - "GET /health HTTP/1.1" 200 OK


2026-10-30 15:44:59.211 127.0.0.1:11965 - "GET /health HTTP/1.1" 200


[테스트 3] http://localhost:8000 -> ['서버 연결됨']


코드를 고치지 않고 환경 변수 값만 바꿨는데 사이드바가 "서버 연결 실패"와 "서버 연결됨"으로 갈렸습니다. 사내에서는 대시보드를 띄우는 설정에 이 값을 적습니다.

---

### 5.4 대시보드 코드 해부

대시보드에서 사용한 핵심 패턴을 정리합니다.

```
패턴                     코드                              용도
──────────────────────────────────────────────────────────────────
API 호출                 call_api(url, json_data)          FastAPI와 통신
Base64 인코딩            base64.b64encode(bytes)           이미지를 JSON에 담기
에러 처리                try/except → st.error()           실패 시 사용자 안내
상태 유지                st.session_state["last_result"]   재실행 시 결과 유지
사이드바                 with st.sidebar:                  설정/옵션 배치
컬럼 레이아웃            col1, col2 = st.columns(2)        입력/결과 나란히
메트릭 표시              st.metric(label, value)           예측 결과 강조
확률 바                  st.progress(value, text)          확률 분포 시각화
```

> `st.session_state`를 사용하는 이유:
> Streamlit은 버튼을 누를 때마다 스크립트를 재실행합니다.
> 결과를 `session_state`에 저장하지 않으면, 재실행 시 결과가 사라집니다.

---

### 5.5 프로젝트 구조 확인

```
model-serving-course/
├── app/                                 ← 백엔드 (FastAPI)
│   ├── error_handlers.py                ← 비동기 처리 노트
│   ├── logger_config.py                 ← 비동기 처리 노트
│   ├── main_final.py                    ← 비동기 처리 노트
│   ├── middleware.py                    ← 비동기 처리 노트
│   ├── model_utils.py                   ← REST API 노트
│   └── schemas.py                       ← FastAPI 기초 노트
├── frontend/                             ← 프론트엔드 (Streamlit)
│   └── app_dashboard.py                 ← 5장: 대시보드
├── models/
│   └── mnist_state_dict.pth
├── .gitignore
└── requirements.txt
```

> FastAPI 기초 노트와 비동기 처리 노트에서 만들었던 `api_client.py`, `ui_helpers.py` 등 별도 모듈 없이,
> 대시보드 파일 하나로 완결됩니다.
> 코드가 커지면 정형 데이터 프로젝트 부록에서 모듈 분리를 도입합니다.

---

### 5장까지의 체크포인트

```
[섹션 1: Streamlit 소개]
Q1. Streamlit의 스크립트 재실행 모델이란?

[섹션 2: 핵심 컨셉]
Q2. @st.cache_resource를 사용하는 이유는?

[섹션 3: System Architecture]
Q3. 프론트엔드와 백엔드를 분리하는 핵심 이유 두 가지는?
Q4. Streamlit 앱에 PyTorch가 필요 없는 이유는?

[섹션 4: API 호출]
Q5. API 호출 실패 시 사용자에게 스택 트레이스가 아닌 메시지를 보여줘야 하는 이유는?

[섹션 5: 실습]
Q6. st.session_state에 결과를 저장하는 이유는?
Q7. 이미지를 API로 전달할 때 Base64 인코딩이 필요한 이유는?
```

---

### 1\~5장 요약

```
오늘 한 일:
  - Streamlit의 기본 개념과 실행 모델을 이해했습니다.
  - 프론트엔드-백엔드 분리 아키텍처의 이유와 구조를 이해했습니다.
  - Streamlit에서 FastAPI를 호출하고 에러를 처리하는 패턴을 익혔습니다.
  - MNIST 추론 대시보드를 완성했습니다.

이어서 할 일 (6장부터):
  - MNIST 추론 API에 API Key 인증을 적용합니다.
  - 이미지 파일 업로드에 안전장치를 더한 이미지 분류 API를 완성합니다.
```

---

> **마치며**
>
> 지금까지 MNIST라는 예제 데이터셋으로 연습했습니다.  
> 6장부터는 이 MNIST 추론 API에 **API Key 인증**과 **파일 업로드**를 더합니다.

1\~5장에서는 API 서버 앞에 화면을 붙였습니다. 6장부터는 API 서버 쪽으로 돌아가 API Key 인증을 걸고 이미지를 파일로 받습니다.

# 인증 및 미디어 처리 기초

## 6. API 보안의 필요성: 왜 아무나 호출하면 안 되는가?

---

> **학습 목표**
> - 인증 없는 API의 위험을 구체적인 시나리오로 이해합니다.
> - API 인증 방식의 종류를 파악하고, 이 과정에서 사용할 방식을 알 수 있습니다.

---

### 6.1 지금까지 만든 API의 문제

1\~5장에서 대시보드에 붙인 MNIST 추론 API는 **누구나 호출할 수 있습니다.**
URL만 알면 세상 누구든 추론 요청을 보낼 수 있다는 뜻입니다.

이것이 왜 문제인지 구체적인 시나리오를 살펴보겠습니다.

![image.png](lms-img:0)

```
[시나리오 1: 서버 다운]
  누군가 while True: requests.post("/predict", ...) 를 실행합니다.
  초당 수백 건의 요청이 쏟아집니다.
  모델 추론은 CPU/GPU를 소비하므로, 서버가 과부하로 다운됩니다.
  → 정상 사용자도 서비스를 이용할 수 없게 됩니다.
```

```
[시나리오 2: 비용 폭탄]
  GPU 서버를 클라우드(AWS, GCP)에서 운영 중입니다.
  인증 없이 누구나 호출 가능하면, 불특정 다수의 요청으로 GPU 사용량이 치솟습니다.
  → 월말 청구서에 예상치 못한 금액이 찍힙니다.
```

```
[시나리오 3: 모델 도용]
  공격자가 대량의 입력-출력 쌍을 수집합니다.
  이 데이터로 여러분의 모델을 모방하는 모델을 학습합니다 (Model Stealing).
  → 여러분이 만든 모델의 가치가 유출됩니다.
```

```
[시나리오 4: 추적 불가]
  에러가 발생했을 때, "누가 어떤 요청을 보냈는지" 알 수 없습니다.
  → 문제 원인을 파악할 수 없고, 악의적 사용자를 차단할 수 없습니다.
```

> 인증은 "누가 호출하는지 확인"하고, "허용된 사용자만 통과"시키는 것입니다.

---

### 6.2 이 과정에서 사용할 인증 방식

인증 방식은 여러 가지가 있습니다. **API Key**는 요청 헤더에 키를 포함하는 가장 간단한 방식이고, **JWT**(JSON Web Token)는 로그인 후 발급된 토큰을 사용하는 방식, **OAuth 2.0**은 Google이나 GitHub 같은 외부 서비스로 인증을 위임하는 방식입니다. 복잡도와 보안 수준은 API Key < JWT < OAuth 순으로 올라갑니다.

ML 추론 API에서는 **API Key 방식**이 가장 흔합니다. OpenAI, Anthropic, Hugging Face 등 대부분의 ML API가 이 방식을 사용합니다. 사용자 로그인이 필요 없고, 키 하나로 인증과 사용량 추적이 모두 가능하기 때문입니다.

> 이 과정에서는 **API Key 방식**만 다룹니다. 가장 간단하면서도 실무적으로 충분합니다.

---

### 체크포인트

1. 인증 없는 API가 위험한 이유를 두 가지 이상 설명하세요.
2. API Key 방식이 ML 추론 API에 적합한 이유는 무엇입니까?

---

> **다음 섹션에서는** FastAPI에 API Key 인증을 직접 구현합니다.

## 7. API Key 인증 구현: Header 기반 인증

---

> **학습 목표**
> - FastAPI에서 API Key 인증을 구현할 수 있습니다.
> - 인증이 실패하면 401 에러를 반환하는 구조를 만들 수 있습니다.
> - `Depends()`로 엔드포인트에 인증을 주입하는 패턴을 이해합니다.

---

### 7.1 API Key 인증의 동작 원리

![image.png](lms-img:0)

```
[인증 성공 흐름]

클라이언트:
  POST /predict
  Headers:
    X-API-Key: test-key-001          ← 요청 헤더에 키 포함

서버:
  1. X-API-Key 헤더에서 값을 추출합니다.
  2. 등록된 키 목록에 있는지 확인합니다.
  3. 있으면 → "사용자A"라는 정보와 함께 엔드포인트 함수를 실행합니다.
```

```
[인증 실패 흐름]

클라이언트:
  POST /predict
  Headers:
    X-API-Key: wrong-key             ← 잘못된 키

서버:
  1. X-API-Key 헤더에서 값을 추출합니다.
  2. 등록된 키 목록에 없습니다.
  3. 401 Unauthorized를 반환합니다. 엔드포인트 함수는 실행되지 않습니다.
```

---

### 7.2 인증 모듈 구현

In [4]:
%%writefile app/auth.py
"""
Day 6 - API Key 인증
"""
from fastapi import HTTPException, Header

# API Key 설정
# 실무에서는 환경 변수나 시크릿 매니저에서 로드합니다.
# 여기서는 학습 목적으로 하드코딩합니다.
VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:  # *your code* — Header에서 키 추출
    """
    API Key를 검증합니다.

    동작 원리:
      - FastAPI는 파라미터 이름 x_api_key를 HTTP 헤더 X-API-Key에 자동 매핑합니다.
        (언더스코어 _ → 하이픈 -, 대소문자 무시)
      - Header(None): 헤더가 없으면 None이 들어옵니다.

    Returns:
        인증된 사용자 이름 (예: "사용자A")
    Raises:
        HTTPException(401): 키가 없거나 유효하지 않을 때
    """
    if x_api_key is None:
        raise HTTPException(
            status_code=401,                       # *your code* — Unauthorized
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )

    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )

    return VALID_API_KEYS[x_api_key]  # 사용자 이름 반환

Writing app/auth.py


> `Header(None)`이 하는 일:
> - 요청의 `X-API-Key` 헤더 값을 `x_api_key` 변수에 넣습니다.
> - 헤더가 없으면 `None`이 됩니다.
> - FastAPI가 자동으로 `x_api_key` → `X-API-Key`로 매핑합니다.

---

### 7.3 엔드포인트에 인증 적용

인증을 적용하는 방법은 **`Depends(verify_api_key)` 한 줄을 추가**하는 것입니다.

> **주의** : **아래 코드는 실행하지 마세요.**  
> `Depends(verify_api_key)`가 엔드포인트에 어떻게 적용되는지 보여주는 구조 설명용 코드입니다.  
> 실제 서버 코드는 섹션 11에서 작성합니다.

```python
from fastapi import FastAPI, Depends
from app.auth import verify_api_key

app = FastAPI()

# 인증 없는 엔드포인트 (기존)
@app.get("/health")
async def health():
    return {"status": "healthy"}

# 인증이 적용된 엔드포인트
@app.post("/predict")
async def predict(
    request: PredictRequest,
    user: str = Depends(verify_api_key),    # ← 이 한 줄이 인증을 적용합니다
):
    # user에는 인증된 사용자 이름("사용자A")이 들어옵니다
    print(f"요청한 사용자: {user}")
    ...
```

`Depends(verify_api_key)`가 하는 일을 단계별로 보면:

```
1. 요청이 들어옵니다.
2. FastAPI가 predict() 함수를 실행하기 전에, verify_api_key()를 먼저 실행합니다.
3. verify_api_key()가 성공하면 → 반환값("사용자A")이 user 변수에 들어갑니다.
   verify_api_key()가 HTTPException을 발생시키면 → predict()는 실행되지 않고 401이 반환됩니다.
```

> `Depends()`는 FastAPI의 **의존성 주입** 기능입니다.
> "이 함수를 실행하기 전에, 먼저 이것을 실행해라"라는 의미입니다.
> 인증 외에도 DB 커넥션 주입, 권한 확인 등 다양하게 활용됩니다.

---

### 7.4 인증 모듈 단독 테스트

서버 없이 인증 로직만 먼저 테스트합니다.

In [5]:
from app.auth import VALID_API_KEYS, verify_api_key
import asyncio

print("등록된 API Key:")
for key, user in VALID_API_KEYS.items():
    print(f"  {key} → {user}")

등록된 API Key:
  test-key-001 → 사용자A
  test-key-002 → 사용자B


In [6]:
# 올바른 키로 테스트
result = await verify_api_key(x_api_key="test-key-001")
print(f"\n올바른 키 → 사용자: {result}")


올바른 키 → 사용자: 사용자A


In [7]:
# 잘못된 키로 테스트
from fastapi import HTTPException

try:
    await verify_api_key(x_api_key="wrong-key")
except HTTPException as e:
    print(f"잘못된 키 → {e.status_code}: {e.detail}")

잘못된 키 → 401: 유효하지 않은 API Key입니다.


In [8]:
# 키 없이 테스트
try:
    await verify_api_key(x_api_key=None)
except HTTPException as e:
    print(f"키 없음 → {e.status_code}: {e.detail}")

키 없음 → 401: API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.


> 서버를 실행하지 않아도 인증 로직을 테스트할 수 있습니다.
> 실제 서버에서의 통합 테스트는 섹션 11에서 수행합니다.

---

### 체크포인트

1. `Header(None)`에서 `None`은 어떤 상황에서 `x_api_key`에 들어갑니까?
2. `Depends(verify_api_key)`를 엔드포인트에 추가하면 요청 처리 흐름이 어떻게 바뀝니까?
3. 인증 실패 시 반환하는 HTTP 상태 코드 401의 의미는 무엇입니까?

---

> **다음 섹션에서는** 정형 데이터를 넘어
> 이미지와 텍스트 같은 비정형 데이터를 API에서 처리하는 방법을 배웁니다.

## 8. 비정형 데이터의 세계: 이미지와 텍스트 입력 처리

---

> **학습 목표**
> - 정형 데이터와 비정형 데이터의 차이를 이해합니다.
> - FastAPI의 `UploadFile`을 사용하여 파일 업로드를 구현합니다.

---

### 8.1 정형 vs 비정형 데이터

정형 데이터 프로젝트 부록에서는 숫자 8개를 JSON으로 보냅니다. 크기가 고정되어 있고, 데이터 형식이 단순합니다.
하지만 이미지나 텍스트는 다릅니다.

```
정형 데이터 프로젝트 부록 (정형 데이터):
  입력: {"MedInc": 3.5, "HouseAge": 25, ...}
  크기: 항상 JSON 몇 백 바이트
  처리: 정규화 → 텐서 변환

챗봇 프로젝트 부록 (비정형 데이터):
  입력: 이미지 파일 (PNG, JPEG) 또는 텍스트 문자열
  크기: 수 KB ~ 수 MB (가변)
  처리: 파일 검증 → 디코딩 → 리사이징 → 정규화 → 텐서 변환
```

비정형 데이터는 처리 과정이 더 복잡하고, **검증 단계가 추가**됩니다.
파일이 진짜 이미지인지, 크기가 적절한지, 읽을 수 있는지 확인해야 합니다.

---

### 8.2 이미지 전송 방식: Base64 vs UploadFile

지금까지는 이미지를 Base64로 인코딩하여 JSON에 담아 보냈습니다.
오늘은 **UploadFile**을 사용합니다. 파일을 직접 보내는 방식이고, Swagger UI에서 파일 선택 버튼이 자동으로 나타나 테스트가 편합니다.

두 방식의 판단 기준은 간단합니다. 다른 JSON 데이터와 함께 이미지를 하나의 요청에 담아야 하면 Base64, 파일을 독립적으로 업로드하면 UploadFile입니다. 실무에서는 UploadFile 방식이 더 자주 사용됩니다.

---

### 8.3 FastAPI의 UploadFile 기본 사용법

In [ ]:
# UploadFile의 핵심 속성과 메서드를 이해합니다.
# 이 코드는 서버 코드이므로 직접 실행하지 않고, 구조만 파악합니다.

"""
from fastapi import FastAPI, UploadFile, File

app = FastAPI()

@app.post("/upload")
async def upload_image(
    file: UploadFile = File(..., description="업로드할 이미지 파일"),
):
    # file 객체의 주요 속성
    file.filename      # "cat.png" — 원본 파일 이름
    file.content_type   # "image/png" — MIME 타입
    file.size           # 12345 — 파일 크기 (바이트)

    # 파일 내용 읽기 (비동기)
    contents = await file.read()   # bytes 타입으로 반환

    return {"filename": file.filename, "size": len(contents)}
"""

print("UploadFile의 핵심:")
print("  file.filename     → 파일 이름")
print("  file.content_type → MIME 타입 (image/png 등)")
print("  await file.read() → 파일 내용 (bytes)")

### 8.3 FastAPI의 UploadFile 기본 사용법

아래는 구조 설명용 코드입니다. **(실행 X, 서버 코드는 섹션 11에서 작성합니다)**

```python
from fastapi import FastAPI, UploadFile, File

app = FastAPI()

@app.post("/upload")
async def upload_image(
    file: UploadFile = File(..., description="업로드할 이미지 파일"),
):
    file.filename      # "cat.png" : 원본 파일 이름
    file.content_type   # "image/png" : MIME 타입

    contents = await file.read()   # bytes 타입으로 반환

    return {"filename": file.filename, "size": len(contents)}
```

| 속성/메서드 | 설명 | 예시 |
|---|---|---|
| `file.filename` | 파일 이름 | `'cat.png'` |
| `file.content_type` | MIME 타입 | `'image/png'` |
| `await file.read()` | 파일 내용 (bytes) | `b'\x89PNG...'` |

> `UploadFile`을 사용하면 Swagger UI에서 **파일 선택 버튼**이 자동으로 나타납니다.  
> Base64처럼 인코딩 문자열을 직접 붙여넣을 필요가 없습니다.  
> 실제 서버 코드에서의 사용은 섹션 11에서 직접 구현합니다.

---

## 9. 파일 업로드와 안전장치: 크기 제한, 이미지 리사이징

---

> **학습 목표**
> - 파일 업로드 시 크기 제한, 형식 검증 등 안전장치를 구현할 수 있습니다.
> - 이미지를 모델 입력에 맞게 리사이징하고 변환할 수 있습니다.

---

### 9.1 안전장치가 필요한 이유

인증으로 "누가 호출하는지"를 확인했지만, 인증된 사용자도 잘못된 파일을 보낼 수 있습니다.

![image.png](lms-img:0)

```
위험한 입력 예시:

100MB 이미지         → 서버 메모리 부족, 다른 사용자에게 영향
.exe를 .png로 위장    → 이미지로 열려고 하면 에러 또는 보안 위험
10000x10000 픽셀     → 전처리에 과도한 시간, 서버 응답 지연
손상된 파일           → PIL.Image.open()에서 예외 발생
```

이런 입력을 모델에 도달하기 전에 차단해야 합니다.

---

### 9.2 안전장치 구현

4가지 검증을 순서대로 수행하는 함수를 만듭니다.

In [9]:
%%writefile app/image_utils.py
"""
이미지 업로드 안전장치 + 전처리
"""
from fastapi import UploadFile, HTTPException
from PIL import Image
import io

# 허용 설정
ALLOWED_TYPES = {"image/png", "image/jpeg", "image/jpg"}
MAX_FILE_SIZE = 5 * 1024 * 1024  # 5MB                     # *your code* — 최대 파일 크기


async def validate_and_read_image(
    file: UploadFile,
    max_size: int = MAX_FILE_SIZE,
    target_size: tuple = (28, 28),
) -> Image.Image:
    """
    업로드된 파일을 검증하고, PIL 이미지로 반환합니다.

    검증 순서:
      1. 파일 타입 검증 → 허용된 형식(PNG, JPEG)만 통과
      2. 파일 크기 검증 → 5MB 이하만 통과
      3. 이미지 디코딩 검증 → 실제로 열 수 있는 이미지만 통과
      4. 리사이징 + 그레이스케일 변환 → 모델 입력 크기에 맞춤
    """

    # ─── 1. 파일 타입 검증 ─────────────────────────
    # content_type은 클라이언트가 보낸 MIME 타입입니다.
    # .exe를 .png로 위장해도 content_type이 다르므로 차단됩니다.
    if file.content_type not in ALLOWED_TYPES:               # *your code* — 타입 체크
        raise HTTPException(
            status_code=400,
            detail=f"지원하지 않는 파일 형식입니다: {file.content_type}. "
                   f"허용 형식: {ALLOWED_TYPES}",
        )

    # ─── 2. 파일 크기 검증 ─────────────────────────
    # 파일 전체를 읽어서 크기를 확인합니다.
    # 이 시점에서 파일 내용이 메모리에 올라옵니다.
    contents = await file.read()
    if len(contents) > max_size:                             # *your code* — 크기 체크
        raise HTTPException(
            status_code=400,
            detail=f"파일 크기가 {max_size // (1024*1024)}MB를 초과합니다. "
                   f"현재: {len(contents) / (1024*1024):.1f}MB",
        )

    # ─── 3. 이미지 디코딩 검증 ─────────────────────
    # content_type이 image/png여도 파일 내용이 실제로 이미지가 아닐 수 있습니다.
    # PIL로 열어보면서 확인합니다.
    try:
        image = Image.open(io.BytesIO(contents))
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="이미지를 읽을 수 없습니다. 파일이 손상되었을 수 있습니다.",
        )

    # ─── 4. 리사이징 + 그레이스케일 변환 ──────────────
    # 어떤 크기의 이미지가 들어와도 모델 입력에 맞게 변환합니다.
    image = image.convert("L").resize(target_size)           # *your code* — 그레이스케일 + 리사이즈

    return image

Writing app/image_utils.py


각 검증 단계를 정리합니다:

```
단계                 차단하는 위험            실패 시 상태 코드
────────────────────────────────────────────────────────
1. 타입 검증          .exe 위장, .txt 전송     400
2. 크기 검증          100MB 이미지             400
3. 디코딩 검증        손상된 파일               400
4. 리사이징          10000x10000 이미지       (자동 변환)
```

> 4번은 에러를 발생시키지 않고, 어떤 크기의 이미지든 모델 입력에 맞게 변환합니다.
> 모든 검증을 통과한 이미지만 모델에 도달합니다.

---

### 9.3 안전장치 단독 테스트

서버 없이 `validate_and_read_image`의 로직을 테스트합니다.

In [10]:
from PIL import Image
import io

# 테스트 이미지 생성 (28x28 그레이스케일)
test_img = Image.new("L", (100, 100), color=128)   # 100x100 회색 이미지
buffer = io.BytesIO()
test_img.save(buffer, format="PNG")
test_bytes = buffer.getvalue()

print(f"테스트 이미지 크기: {len(test_bytes)} bytes")
print(f"테스트 이미지 해상도: {test_img.size}")

테스트 이미지 크기: 156 bytes
테스트 이미지 해상도: (100, 100)


In [11]:
# 리사이징 테스트
img = Image.open(io.BytesIO(test_bytes))
img_resized = img.convert("L").resize((28, 28))
print(f"변환 전: {img.size} → 변환 후: {img_resized.size}")

변환 전: (100, 100) → 변환 후: (28, 28)


In [12]:
# 잘못된 파일 테스트
try:
    Image.open(io.BytesIO(b"this is not an image"))
except Exception as e:
    print(f"디코딩 실패: {type(e).__name__}")
# → 이런 경우 validate_and_read_image()가 400 에러를 반환합니다.

디코딩 실패: UnidentifiedImageError


### 9.4 픽셀 수 상한

9.2절의 크기 검증은 파일의 **바이트 수**를 봅니다. 그런데 그림 파일은 압축되어 있어서 바이트 수가 작아도 펼치면 클 수 있습니다. 9.1절 그림의 10000x10000 픽셀을 흰 바탕으로 만들어 9.2절 함수에 그대로 넣어 봅니다.

In [13]:
# 9.1절 그림의 "10000x10000 픽셀"을 만들어 9.2절 함수에 그대로 넣어 봅니다
import io
import time

from fastapi import UploadFile
from PIL import Image
from starlette.datastructures import Headers

from app.image_utils import validate_and_read_image


def as_upload(data, filename="upload.png", content_type="image/png"):
    """FastAPI가 엔드포인트에 넘겨주는 것과 같은 UploadFile을 만듭니다."""
    return UploadFile(file=io.BytesIO(data), filename=filename,
                      headers=Headers({"content-type": content_type}))


buf = io.BytesIO()
Image.new("L", (10000, 10000), 255).save(buf, format="PNG")   # 흰 바탕 한 장
big = buf.getvalue()

w, h = Image.open(io.BytesIO(big)).size          # 헤더만 읽습니다
print(f"파일 크기 : {len(big) / 1024:.0f}KB (9.2절 상한 5MB)")
print(f"펼친 크기 : {w}x{h} = {w * h:,}픽셀, 1픽셀 1바이트로 약 {w * h / 1e6:.0f}MB")

t0 = time.time()
image = await validate_and_read_image(as_upload(big))
print(f"9.2절 함수 : 통과, {image.size}로 바뀜, {time.time() - t0:.2f}초")

파일 크기 : 118KB (9.2절 상한 5MB)
펼친 크기 : 10000x10000 = 100,000,000픽셀, 1픽셀 1바이트로 약 100MB
9.2절 함수 : 통과, (28, 28)로 바뀜, 0.13초


d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\PIL\Image.py:3578: DecompressionBombWarning: Image size (100000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


100KB 남짓한 파일이 5MB 상한을 지나 1억 픽셀로 펼쳐진 뒤에야 28x28로 줄었습니다. Pillow도 `DecompressionBombWarning` 경고를 내지만 막지는 않습니다. Pillow는 약 8,950만 픽셀부터 경고하고 그 두 배를 넘을 때 오류를 냅니다.

`Image.open()`은 헤더만 읽고 그림을 아직 펼치지 않습니다. 그래서 `size`로 픽셀 수를 먼저 보고 너무 크면 펼치기 전에 돌려보낼 수 있습니다. 9.2절 파일은 그대로 두고 이 검사를 앞에 하나 더 두는 파일을 만듭니다.

In [14]:
%%writefile app/image_utils_safe.py
"""9.2절 image_utils.py 앞에 픽셀 수 검사를 하나 더 둡니다. 원본 파일은 고치지 않습니다."""
from fastapi import HTTPException, UploadFile
from PIL import Image

from app import image_utils

MAX_PIXELS = 4096 * 4096   # 약 1,680만 픽셀. 휴대폰 사진(약 1,200만)은 지나갑니다


async def validate_and_read_image(file: UploadFile, max_pixels: int = MAX_PIXELS, **kwargs):
    """헤더에 적힌 픽셀 수부터 보고, 통과하면 9.2절의 네 단계에 그대로 넘깁니다."""
    try:
        with Image.open(file.file) as head:    # 헤더만 읽고 아직 펼치지 않습니다
            width, height = head.size
    except Exception:
        width = height = 0                      # 열리지 않는 파일은 9.2절 3단계가 400으로 막습니다
    await file.seek(0)                          # 읽은 자리를 처음으로 되돌립니다

    if width * height > max_pixels:
        raise HTTPException(
            status_code=400,
            detail=f"이미지가 너무 큽니다: {width}x{height}. 최대 {max_pixels:,}픽셀까지 받습니다.",
        )
    return await image_utils.validate_and_read_image(file, **kwargs)

Writing app/image_utils_safe.py


In [15]:
from fastapi import HTTPException

from app.image_utils_safe import validate_and_read_image as safe_read

buf = io.BytesIO()
Image.new("L", (4000, 3000), 255).save(buf, format="PNG")     # 휴대폰 사진 크기

cases = [
    ("4000x3000", buf.getvalue(), "image/png"),
    ("10000x10000", big, "image/png"),
    (".exe를 image/png라고 보냄", b"MZ" + bytes(200), "image/png"),
]
for name, data, ctype in cases:
    try:
        image = await safe_read(as_upload(data, content_type=ctype))
        print(f"{name} -> 통과, {image.size}")
    except HTTPException as e:
        print(f"{name} -> {e.status_code} {e.detail}")

4000x3000 -> 통과, (28, 28)
10000x10000 -> 400 이미지가 너무 큽니다: 10000x10000. 최대 16,777,216픽셀까지 받습니다.
.exe를 image/png라고 보냄 -> 400 이미지를 읽을 수 없습니다. 파일이 손상되었을 수 있습니다.


휴대폰 사진 크기는 지나가고 1억 픽셀은 펼치기 전에 400으로 막혔습니다. `content_type`은 보내는 쪽이 적는 값이라 `.exe` 파일도 `image/png`라고 적어 보낼 수 있습니다. 그런 파일은 1단계를 지나 3단계의 디코딩에서 막힙니다.

🔎 **더 알아보기**
5장 대시보드 그림의 업로드 칸에 적힌 `Limit 200MB per file`은 Streamlit의 기본 상한입니다. API 서버가 5MB에서 거절한다면 `.streamlit/config.toml`의 `[server]`에 `maxUploadSize = 5`를 적어 큰 파일이 대시보드에서 먼저 걸리게 합니다. 두 상한은 같은 환경 변수에서 읽게 해 두어야 한쪽만 고치는 일이 없습니다.

---

### 체크포인트

1. `UploadFile`과 Base64 방식의 핵심 차이는 무엇입니까?
2. `file.content_type`으로 타입을 검증하는 이유는 무엇입니까?
3. 파일 크기를 제한하지 않으면 어떤 문제가 발생할 수 있습니까?

---

> **다음 섹션에서는** 인증 + 파일 업로드 + 모델 추론을 결합한 API를 완성합니다.

## 10. 추론 파이프라인: 전처리 → 모델 호출 → 후처리

---

> **학습 목표**
> - 이미지 업로드 → 전처리 → 모델 추론 → 결과 반환의 전체 파이프라인을 이해합니다.

---

### 10.1 전체 파이프라인

```
[클라이언트]                              [서버]

POST /predict/image
Headers: X-API-Key: test-key-001     1. API Key 인증 (섹션 7)
Body: multipart/form-data            2. 파일 검증 + 리사이징 (섹션 9)
  └─ file: digit.png                 3. 텐서 변환 + 정규화
         ──────────────▶             4. 모델 추론 (run_in_executor)
                                     5. 후처리
         ◀──────────────             6. JSON 응답
{
  "predicted_class": "7",
  "confidence": 0.98
}
```

> 각 단계는 지금까지 배운 기술의 조합입니다.
> 새로운 것은 1번(인증)과 2번(파일 검증)뿐이고, 나머지는 이미 익숙한 패턴입니다.

---

## 11. 실습: 이미지를 업로드하면 분류 결과를 반환하는 API 만들기

---

> **실습 목표**
> - API Key 인증 + UploadFile + MNIST 모델을 결합한 API를 완성합니다.
> - Swagger UI에서 이미지를 업로드하여 추론 결과를 확인합니다.

---

### 11.1 통합 API 서버 코드

In [16]:
%%writefile app/image_api.py
"""
Day 6 - 이미지 분류 API (인증 + 파일 업로드 + MNIST 모델)
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
from fastapi import FastAPI, UploadFile, File, Depends, HTTPException
from torchvision import transforms

from app.model_utils import load_model, predict, CLASS_NAMES
from app.auth import verify_api_key
from app.image_utils import validate_and_read_image
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("image_api")

app = FastAPI(
    title="Image Classification API",
    description="이미지를 업로드하면 숫자(0~9)를 분류하는 API (인증 필요)",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="image")

# ===== 모델 로드 =====
MODEL_PATH = "models/mnist_state_dict.pth"
model = None

# 전처리 파이프라인 (PIL → Tensor)
img_transform = transforms.Compose([               # *your code* — transforms.Compose 구성
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


@app.on_event("startup")
async def startup():
    global model
    logger.info("MNIST 모델 로드 중...")
    model = load_model(MODEL_PATH)
    logger.info("모델 로드 완료")


def run_inference(tensor: torch.Tensor) -> dict:
    if model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return predict(model, tensor)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy" if model else "loading"}


@app.post("/predict/image", tags=["Inference"])
async def predict_image(
    file: UploadFile = File(..., description="분류할 이미지 (PNG, JPEG)"),
    user: str = Depends(verify_api_key),                     # *your code* — 인증 적용
):
    """
    이미지를 업로드하면 숫자(0~9)를 분류합니다.
    X-API-Key 헤더에 유효한 API Key가 필요합니다.
    """
    logger.info(f"추론 요청 — 사용자: {user}, 파일: {file.filename}")

    # 1. 파일 검증 + 이미지 로드 (28x28 그레이스케일로 변환)
    image = await validate_and_read_image(file, target_size=(28, 28))  # *your code* — 안전장치 적용

    # 2. 텐서 변환
    tensor = img_transform(image).unsqueeze(0)   # (1, 1, 28, 28)

    # 3. 비동기 추론
    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(executor, run_inference, tensor)
    except Exception as e:
        logger.error(f"추론 실패: {e}")
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    logger.info(f"추론 완료 — 결과: {result['predicted_class']}, 확신도: {result['confidence']:.2f}")

    return {
        "success": True,
        "predicted_class": result["predicted_class"],
        "confidence": round(result["confidence"], 4),
        "user": user,
    }

Writing app/image_api.py


🔎 **더 알아보기**
`@app.on_event("startup")`은 이 노트의 버전(FastAPI 0.141)에서 폐기 예정이라 `DeprecationWarning`을 냅니다. 서버 스레드 안에서 나는 경고라 코랩 출력에는 보이지 않고 서버는 그대로 돕니다. 새로 쓰는 코드라면 FastAPI 문서의 lifespan 방식을 씁니다.

---

### 11.2 서버 실행

In [17]:
# CLASS_NAMES가 model_utils.py에 없으면 추가합니다
with open("app/model_utils.py", "r", encoding="utf-8") as f:
    content = f.read()

if "CLASS_NAMES" not in content:
    with open("app/model_utils.py", "a", encoding="utf-8") as f:
        f.write("\n\n# MNIST 클래스 이름\nCLASS_NAMES = [str(i) for i in range(10)]\n")
    print("✅ CLASS_NAMES 추가 완료")
else:
    print("✅ CLASS_NAMES 이미 있음")

✅ CLASS_NAMES 이미 있음


🖥 코랩에서 세션을 다시 시작했다면 **코랩에서 할 때** 셀부터 실행한 뒤 서버를 띄웁니다. 건너뛰면 작업 폴더가 `/content`라서 서버 스레드가 `ModuleNotFoundError: No module named 'app'`로 멈추는데도 서버 셀은 성공 문구를 찍습니다.

In [ ]:
# ⚠️ 이전 섹션에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel 후 이 셀부터 실행

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.image_api:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨 — http://localhost:8000/docs 에서 Swagger UI를 확인하세요.")

✅ 서버 시작됨 — http://localhost:8000/docs 에서 Swagger UI를 확인하세요.


INFO:     Started server process [7964]
INFO:     Waiting for application startup.


2026-10-30 16:50:24 INFO     [image_api] MNIST 모델 로드 중...
2026-10-30 16:50:24 INFO     [image_api] 모델 로드 완료


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔎 **더 알아보기**
이 셀은 서버를 띄운 뒤 3초만 기다립니다. 서버가 첫 `import torch`에 그보다 오래 걸리면 11.3절의 첫 요청이 `ConnectionError`로 끝납니다(이 노트의 검증 환경에서는 3.6초가 걸려 실제로 그랬습니다). 그때는 출력에 `Uvicorn running`이 찍힌 뒤 11.3절 셀을 다시 실행하십시오.

출력에 `No module named 'app'`이 있으면 서버는 뜨지 않았습니다. **코랩에서 할 때** 셀을 먼저 실행한 뒤 11.2절 서버 셀을 다시 실행하십시오.

---

### 11.3 테스트 1: 인증 없이 요청 → 401

In [2]:
import requests

# API Key 없이 요청
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.png", b"fake image data", "image/png")},
    # headers 없음 → 인증 실패
)

print(f"상태 코드: {response.status_code}")   # 401
print(f"응답: {response.json()}")

POST /predict/image → 401 (0.001s)


INFO:     127.0.0.1:1039 - "POST /predict/image HTTP/1.1" 401 Unauthorized
상태 코드: 401
응답: {'detail': 'API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.'}


---

### 11.4 테스트 2: 잘못된 키 → 401

In [3]:
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.png", b"fake image data", "image/png")},
    headers={"X-API-Key": "wrong-key"},                      # *your code* — 잘못된 키
)

print(f"상태 코드: {response.status_code}")   # 401
print(f"응답: {response.json()}")

POST /predict/image → 401 (0.001s)


INFO:     127.0.0.1:1043 - "POST /predict/image HTTP/1.1" 401 Unauthorized
상태 코드: 401
응답: {'detail': '유효하지 않은 API Key입니다.'}


---

### 11.5 테스트 3: 올바른 키 + MNIST 이미지 → 성공

In [4]:
with open("app/image_api.py", "r", encoding="utf-8") as f:
    content = f.read()

# predicted_class → label 로 교체
if "predicted_class" in content:
    content = content.replace("predicted_class", "label")
    with open("app/image_api.py", "w", encoding="utf-8") as f:
        f.write(content)
    print("✅ predicted_class → label 로 수정 완료")
else:
    print("✅ 이미 수정되어 있음")

✅ predicted_class → label 로 수정 완료


🖥 코랩에서 세션을 다시 시작했다면 **코랩에서 할 때** 셀부터 실행한 뒤 서버를 띄웁니다. 건너뛰면 작업 폴더가 `/content`라서 서버 스레드가 `ModuleNotFoundError: No module named 'app'`로 멈추는데도 서버 셀은 성공 문구를 찍습니다.

In [ ]:
import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.image_api:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨")

✅ 서버 시작됨


INFO:     Started server process [21764]
INFO:     Waiting for application startup.


2026-10-30 16:52:24 INFO     [image_api] MNIST 모델 로드 중...
2026-10-30 16:52:24 INFO     [image_api] 모델 로드 완료


INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [2]:
try:
    from app import image_api
    print("✅ import 성공")
except Exception as e:
    print(f"❌ import 실패: {e}")

✅ import 성공


In [3]:
with open("app/image_api.py", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if "label" in line.lower() or "predict" in line.lower():
            print(f"{i}: {line.rstrip()}")

11: from app.model_utils import load_model, predict, CLASS_NAMES
55:     return predict(model, tensor)
65: @app.post("/predict/image", tags=["Inference"])
66: async def predict_image(
90:     logger.info(f"추론 완료 — 결과: {result['label']}, 확신도: {result['confidence']:.2f}")
94:         "label": result["label"],


In [4]:
from torchvision import datasets
from PIL import Image
import io
import requests

# MNIST 테스트 이미지 가져오기
test_dataset = datasets.MNIST(root="data", train=False, download=True)
test_image, test_label = test_dataset[0]   # 첫 번째 테스트 이미지

# PIL 이미지 → bytes 변환
buf = io.BytesIO()
test_image.save(buf, format="PNG")
image_bytes = buf.getvalue()

print(f"테스트 이미지 정답: {test_label}")

# API 호출
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("digit.png", image_bytes, "image/png")},
    headers={"X-API-Key": "test-key-001"},                   # *your code* — 올바른 키
)

print(f"상태 코드: {response.status_code}")   # 200
result = response.json()
print(f"예측 결과: {result}")

테스트 이미지 정답: 7
2026-10-30 16:52:32 INFO     [image_api] 추론 요청 — 사용자: 사용자A, 파일: digit.png


에러 발생: KeyError: 'label'
경로: POST http://localhost:8000/predict/image
스택 트레이스:
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\errors.py", line 164, in __call__
    await self.app(scope, receive, _send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 203, in __call__
    response = await self.dispatch_func(request, call_next)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\app\middleware.py", line 16, in dispatch
    response = await call_next(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 179, in call_next
    raise app_exc from app_exc.__cause__ or app_exc.__context__
  File "d:\vscode\aiffel\2026\10\27\model-serving-course

INFO:     127.0.0.1:8928 - "POST /predict/image HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\fastapi\applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "d:\vscode\aiffel\2026\10

상태 코드: 500
예측 결과: {'success': False, 'error': '서버 내부 오류가 발생했습니다.'}


🔎 **더 알아보기**
`상태 코드: 500`이 나왔다면 `app/model_utils.py`의 `predict()`가 `label` 대신 `predicted_class`를 돌려주는 버전입니다(5장 대시보드에서 추론 결과가 나온 버전). 11.1절 셀을 다시 실행해 `app/image_api.py`를 되돌리고 커널을 재시작하십시오(코랩은 **코랩에서 할 때** 셀부터). 그다음 11.5절의 치환 셀은 건너뛰고 서버 셀부터 실행합니다.

---

### 11.6 테스트 4: 잘못된 파일 형식 → 400

In [5]:
response = requests.post(
    "http://localhost:8000/predict/image",
    files={"file": ("test.txt", b"this is not an image", "text/plain")},
    headers={"X-API-Key": "test-key-001"},
)

print(f"상태 코드: {response.status_code}")   # 400
print(f"응답: {response.json()}")

2026-10-30 16:53:13 INFO     [image_api] 추론 요청 — 사용자: 사용자A, 파일: test.txt


POST /predict/image → 400 (0.002s)


INFO:     127.0.0.1:14318 - "POST /predict/image HTTP/1.1" 400 Bad Request
상태 코드: 400
응답: {'detail': "지원하지 않는 파일 형식입니다: text/plain. 허용 형식: {'image/png', 'image/jpeg', 'image/jpg'}"}


---

### 11.7 테스트 5: 여러 이미지 연속 테스트

In [6]:
import requests
from torchvision import datasets
from PIL import Image
import io

test_dataset = datasets.MNIST(root="data", train=False, download=True)

print("=== 연속 추론 테스트 (5장) ===\n")

for i in range(5):
    img, label = test_dataset[i]

    buf = io.BytesIO()
    img.save(buf, format="PNG")

    resp = requests.post(
        "http://localhost:8000/predict/image",
        files={"file": (f"digit_{i}.png", buf.getvalue(), "image/png")},
        headers={"X-API-Key": "test-key-001"},
    )

    r = resp.json()
    predicted = r.get("predicted_class", "?")
    confidence = r.get("confidence", 0)
    match = "✅" if str(label) == str(predicted) else "❌"

    print(f"  이미지 {i}: 정답={label}, 예측={predicted}, 확신도={confidence:.4f} {match}")

=== 연속 추론 테스트 (5장) ===

2026-10-30 16:53:18 INFO     [image_api] 추론 요청 — 사용자: 사용자A, 파일: digit_0.png


에러 발생: KeyError: 'label'
경로: POST http://localhost:8000/predict/image
스택 트레이스:
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\errors.py", line 164, in __call__
    await self.app(scope, receive, _send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 203, in __call__
    response = await self.dispatch_func(request, call_next)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\app\middleware.py", line 16, in dispatch
    response = await call_next(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 179, in call_next
    raise app_exc from app_exc.__cause__ or app_exc.__context__
  File "d:\vscode\aiffel\2026\10\27\model-serving-course

INFO:     127.0.0.1:14322 - "POST /predict/image HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\fastapi\applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "d:\vscode\aiffel\2026\10

  이미지 0: 정답=7, 예측=?, 확신도=0.0000 ❌
2026-10-30 16:53:20 INFO     [image_api] 추론 요청 — 사용자: 사용자A, 파일: digit_1.png


에러 발생: KeyError: 'label'
경로: POST http://localhost:8000/predict/image
스택 트레이스:
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\errors.py", line 164, in __call__
    await self.app(scope, receive, _send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 203, in __call__
    response = await self.dispatch_func(request, call_next)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\app\middleware.py", line 16, in dispatch
    response = await call_next(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 179, in call_next
    raise app_exc from app_exc.__cause__ or app_exc.__context__
  File "d:\vscode\aiffel\2026\10\27\model-serving-course

... (11 outputs left out)

에러 발생: KeyError: 'label'
경로: POST http://localhost:8000/predict/image
스택 트레이스:
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\errors.py", line 164, in __call__
    await self.app(scope, receive, _send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 203, in __call__
    response = await self.dispatch_func(request, call_next)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\app\middleware.py", line 16, in dispatch
    response = await call_next(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\middleware\base.py", line 179, in call_next
    raise app_exc from app_exc.__cause__ or app_exc.__context__
  File "d:\vscode\aiffel\2026\10\27\model-serving-course

INFO:     127.0.0.1:14330 - "POST /predict/image HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\fastapi\applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "d:\vscode\aiffel\2026\10\27\model-serving-course\.venv_orthia\Lib\site-packages\starlette\applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "d:\vscode\aiffel\2026\10

  이미지 4: 정답=4, 예측=?, 확신도=0.0000 ❌


🔎 **더 알아보기**
11.5절 치환 셀로 응답 키가 `label`이 됐다면 이 셀은 없는 키 `predicted_class`를 읽어 예측을 모두 `?`로 찍습니다. 실제 예측은 서버 로그의 `추론 완료` 줄에 있습니다.

---

### 11.8 Swagger UI에서 테스트

```
1. 브라우저에서 http://localhost:8000/docs 접속
2. POST /predict/image → Try it out
3. x_api_key 입력란에 test-key-001 입력
4. Choose File → 아무 이미지 선택
5. Execute → 결과 확인
```

> `Header()`로 정의한 `x_api_key` 파라미터가 Swagger UI에 입력란으로 나타납니다.

---

🖥 **로컬 작업**
코랩에서는 브라우저가 `localhost:8000`에 닿지 않습니다. FastAPI 기초 노트에서 Swagger UI를 띄운 `serve_kernel_port_as_iframe`을 그대로 쓰고 높이만 넉넉히 줍니다. 이 노트의 검증 환경에는 코랩이 없어 화면이 뜨는 것까지는 확인하지 못했습니다.

In [7]:
# 코랩에서 Swagger UI를 셀 출력 안에 띄웁니다. 코랩이 아니면 주소만 알려 줍니다
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(8000, path="/docs", height="700")
except ImportError:
    print("코랩이 아닙니다. 브라우저에서 http://localhost:8000/docs 로 들어가십시오")

코랩이 아닙니다. 브라우저에서 http://localhost:8000/docs 로 들어가십시오


### 11.9 프로젝트 구조 확인

```
model-serving-course/
├── app/
│   ├── auth.py                ← 7장: API Key 인증
│   ├── image_utils.py         ← 9장: 이미지 검증/전처리
│   ├── image_api.py           ← 11장: 이미지 분류 API (인증 + 업로드)
│   ├── housing_model.py       ← (정형 데이터 프로젝트 부록을 했다면)
│   ├── housing_schemas.py     ← (정형 데이터 프로젝트 부록을 했다면)
│   ├── housing_api.py         ← (정형 데이터 프로젝트 부록을 했다면)
│   ├── model_utils.py         ← REST API 노트
│   ├── schemas.py             ← FastAPI 기초 노트
│   ├── error_handlers.py      ← 비동기 처리 노트
│   ├── logger_config.py       ← 비동기 처리 노트
│   └── middleware.py          ← 비동기 처리 노트
│
├── models/
│   ├── mnist_state_dict.pth
│   ├── housing_model.pth      ← (정형 데이터 프로젝트 부록을 했다면)
│   └── housing_preprocessing.json ← (정형 데이터 프로젝트 부록을 했다면)
└── ...
```

## 12. 사내에 올리기 전에 : 키 보관과 키 해시 목록

### 12.1 키를 둘 자리

7.2절의 `app/auth.py`는 주석에 적힌 대로 학습을 위해 키를 코드에 적었습니다. 이 모양 그대로 사내에 올리면 저장소를 읽을 수 있는 사람은 모두 키를 갖게 됩니다. 키를 바꾸려면 코드를 고쳐 다시 배포해야 하고 한 번 커밋된 키는 지워도 커밋 기록에 남습니다.

**키를 둘 자리**

<div style="font-size: 18px; line-height: 1.6;">

| 자리 | 쓰는 때 | 약점 |
|---|---|---|
| 코드 안 | 쓰지 않습니다 | 저장소를 읽는 사람 모두가 키를 가집니다 |
| 코랩 `userdata` | 수업과 실험 | 코랩 밖에서는 없습니다 |
| 환경 변수 | 사내 서버 한두 대 | 같은 기계의 다른 프로세스와 자식 프로세스가 볼 수 있습니다 |
| 권한 건 파일 | 기계 한 대에서 나만 읽게 둘 때 | 기계가 늘면 관리가 흩어집니다 |
| 사내 시크릿 관리 시스템 | 서버가 여러 대이거나 키를 자주 바꿀 때 | 그 시스템을 따로 운영해야 합니다 |

</div>

같은 코드가 코랩에서도 사내 서버에서도 돌도록 코랩 `userdata`, 환경 변수, 파일 순서로 키를 찾고 하나도 없으면 이 노트에서만 쓸 키를 새로 만듭니다.

In [8]:
import os
import secrets
from pathlib import Path

KEY_TXT = Path("data/my_api_key.txt")


def read_secret(name):
    """키를 코랩 userdata -> 환경 변수 -> 파일 순서로 찾고, 없으면 새로 만들어 파일에 둡니다."""
    try:
        from google.colab import userdata         # 코랩의 보안 비밀
        value = userdata.get(name)
        if value:
            return value, "코랩 userdata"
    except Exception:
        pass                                      # 코랩이 아니거나 등록하지 않았습니다
    if os.environ.get(name):
        return os.environ[name], "환경 변수"
    if KEY_TXT.exists():
        return KEY_TXT.read_text(encoding="utf-8").strip(), "파일"
    KEY_TXT.parent.mkdir(parents=True, exist_ok=True)
    KEY_TXT.write_text("sk-" + secrets.token_hex(16), encoding="utf-8")
    KEY_TXT.chmod(0o600)                          # 나만 읽고 씁니다
    return KEY_TXT.read_text(encoding="utf-8").strip(), "새로 만듦"


MY_KEY, where = read_secret("D49_API_KEY")
print(f"키 출처 : {where}, 앞 6자 : {MY_KEY[:6]}... (전체는 찍지 않습니다)")

키 출처 : 새로 만듦, 앞 6자 : sk-23d... (전체는 찍지 않습니다)


🖥 **로컬 작업**
코랩의 보안 비밀(키 아이콘) 칸에 이름 `D49_API_KEY`로 값을 넣고 노트북 액세스를 켜면 `userdata` 분기로 잡힙니다. 사내 서버에서는 서비스를 띄우는 설정 파일에서 환경 변수로 넣습니다. 셸에 `export`를 손으로 치면 셸 기록에 키가 남습니다.

키는 앞 몇 글자만 찍었습니다. 노트북의 출력도 파일에 저장되어 저장소에 함께 올라갈 수 있기 때문입니다.

### 12.2 키 원문 대신 해시

서버의 키 목록에 키 원문을 적어 두면 키 목록 파일이 새는 순간 모든 키가 함께 샙니다. 그래서 키 목록에는 키의 **해시**만 적습니다. 해시는 `hashlib.sha256`으로 만든 값이라 같은 키는 늘 같은 해시가 되고 해시에서 키를 되돌릴 수는 없습니다.

함수 이름과 헤더는 7.2절과 같아서 `Depends(verify_api_key)` 자리는 그대로 둡니다. 키 목록은 환경 변수 `D49_KEYFILE`이 가리키는 파일에서 읽습니다.

In [9]:
%%writefile app/auth_hashed.py
"""API Key 인증.

키 원문을 코드에도 서버 키 목록에도 두지 않습니다. 키 목록에는 해시만 있고,
들어온 키를 같은 방식으로 해시로 바꿔 맞춰 봅니다.
"""
import hashlib
import hmac
import json
import os
from pathlib import Path

from fastapi import Header, HTTPException

KEY_FILE = Path(os.environ.get("D49_KEYFILE", "data/api_keys.json"))


def fingerprint(key: str) -> str:
    """키를 해시로 바꿉니다. 같은 키는 항상 같은 해시이고, 되돌릴 수 없습니다."""
    return hashlib.sha256(key.encode("utf-8")).hexdigest()


def load_registry():
    """해시 -> 부르는 쪽 이름. 파일이 없으면 빈 키 목록입니다."""
    if not KEY_FILE.exists():
        return {}
    return json.loads(KEY_FILE.read_text(encoding="utf-8"))


async def verify_api_key(x_api_key: str = Header(default=None)) -> str:
    """X-API-Key 헤더를 검사하고, 통과하면 부르는 쪽 이름을 돌려줍니다.

    FastAPI 가 파라미터 이름 x_api_key 를 헤더 X-API-Key 로 바꿔 읽습니다
    (밑줄은 하이픈으로, 대소문자는 구분하지 않습니다).
    """
    if not x_api_key:
        raise HTTPException(
            status_code=401,
            detail="API Key 가 필요합니다. X-API-Key 헤더를 넣어 주십시오",
            headers={"WWW-Authenticate": "X-API-Key"},
        )

    given = fingerprint(x_api_key)
    for known, caller in load_registry().items():
        # 끝까지 비교합니다. 몇 번째 글자에서 틀렸는지가 시간으로 새지 않게 하려는 것입니다
        if hmac.compare_digest(given, known):
            return caller

    raise HTTPException(
        status_code=401,
        detail="등록되지 않은 API Key 입니다",
        headers={"WWW-Authenticate": "X-API-Key"},
    )

Writing app/auth_hashed.py


🔎 **더 알아보기**
`hmac.compare_digest`는 두 값을 끝까지 비교합니다. `==`는 첫 글자가 다르면 바로 멈추므로 몇 번째 글자에서 틀렸는지가 걸린 시간으로 새어 나갈 수 있습니다. 한 줄로 막을 수 있으니 막아 둡니다.

키는 부르는 프로그램마다 따로 발급합니다. 키 목록을 만들고 7.4절처럼 서버 없이 인증 함수만 먼저 테스트합니다.

In [10]:
import json

from fastapi import HTTPException

from app.auth_hashed import KEY_FILE, fingerprint, load_registry, verify_api_key

# 부르는 프로그램마다 키를 따로 발급하고, 키 목록에는 해시만 적습니다
issued = {MY_KEY: "대시보드", "sk-" + secrets.token_hex(16): "야간 일괄 작업"}
KEY_FILE.parent.mkdir(parents=True, exist_ok=True)
KEY_FILE.write_text(json.dumps({fingerprint(k): who for k, who in issued.items()},
                               ensure_ascii=False, indent=1), encoding="utf-8")
KEY_FILE.chmod(0o600)

print("키 목록 :")
for fp, who in load_registry().items():
    print(f"  {fp[:16]}... -> {who}")

print("\n인증 함수 단독 테스트 :")
for label, key in [("내 키", MY_KEY), ("test-key-001", "test-key-001"), ("키 없음", None)]:
    try:
        print(f"  {label} -> 통과, 사용자 : {await verify_api_key(x_api_key=key)}")
    except HTTPException as e:
        print(f"  {label} -> {e.status_code} : {e.detail}")

키 목록 :
  8be8e2f3f2617f05... -> 대시보드
  67a45aec2dfc900a... -> 야간 일괄 작업

인증 함수 단독 테스트 :
  내 키 -> 통과, 사용자 : 대시보드
  test-key-001 -> 401 : 등록되지 않은 API Key 입니다
  키 없음 -> 401 : API Key 가 필요합니다. X-API-Key 헤더를 넣어 주십시오


키 목록 파일에는 키 원문이 없습니다. 7.2절에 적혀 있던 `test-key-001`도 이 키 목록에는 없으므로 401입니다.

키 파일 `data/my_api_key.txt`와 키 목록 `data/api_keys.json`은 프로젝트 폴더 안에 있습니다. 제출하려고 폴더를 깃헙에 올리면 두 파일도 함께 올라가므로 REST API 노트 2.5절에서 만든 `.gitignore`에 적어 둡니다. FastAPI 기초 노트 4.7절의 오류 로그 `server_errors.log`도 같이 적습니다.

In [11]:
# 키 파일과 키 목록, 오류 로그가 저장소에 올라가지 않게 .gitignore에 적습니다
from pathlib import Path

gitignore = Path(".gitignore")
lines = gitignore.read_text(encoding="utf-8").splitlines() if gitignore.exists() else []
for entry in ["data/my_api_key.txt", "data/api_keys.json", "server_errors.log"]:
    if entry not in lines:
        lines.append(entry)
gitignore.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(gitignore.read_text(encoding="utf-8"))

# 가상환경
.venv/

# Python 캐시
__pycache__/
*.pyc
*.pyo

# 모델 파일 (용량이 크므로 Git에 포함하지 않음)
models/*.pth
models/*.onnx
models/*.pt

# 주피터 체크포인트
.ipynb_checkpoints/

# OS 파일
.DS_Store
Thumbs.db
data/my_api_key.txt
data/api_keys.json
server_errors.log



### 12.3 키 해시 목록으로 API 서버 띄우기

11장의 `app/image_api.py`는 그대로 두고 인증을 가져오는 줄만 바꾼 사본을 만듭니다. 11.5절에서 파일 내용을 바꿔 쓴 것과 같은 방법입니다. 키가 목록에 누구로 적혀 있는지 돌려주는 `/whoami`도 하나 더합니다.

In [12]:
# 11장 image_api.py에서 인증을 가져오는 줄만 바꾼 사본을 만듭니다. 원본 파일은 그대로 둡니다
src = Path("app/image_api.py").read_text(encoding="utf-8")
src = src.replace("from app.auth import verify_api_key",
                  "from app.auth_hashed import verify_api_key")
src += '''

@app.get("/whoami", tags=["System"])
async def whoami(user: str = Depends(verify_api_key)):
    """이 키가 키 목록에 누구로 적혀 있는지 돌려줍니다."""
    return {"user": user}
'''
Path("app/image_api_hashed.py").write_text(src, encoding="utf-8")
print("만듦 : app/image_api_hashed.py")

만듦 : app/image_api_hashed.py


In [13]:
import threading
import time

import requests
import uvicorn

# 11장 서버(8000)와 겹치지 않게 8490번에 띄웁니다
hashed_server = uvicorn.Server(uvicorn.Config(
    "app.image_api_hashed:app", host="127.0.0.1", port=8490, log_level="warning"))
hashed_thread = threading.Thread(target=hashed_server.run, daemon=True)
hashed_thread.start()
for _ in range(100):                     # 최대 10초 기다립니다
    if hashed_server.started:
        break
    time.sleep(0.1)

BASE = "http://127.0.0.1:8490"
checks = [
    ("헬스체크, 키 없음", "/health", None),
    ("키 없음", "/whoami", None),
    ("7.2절의 test-key-001", "/whoami", "test-key-001"),
    ("내 키", "/whoami", MY_KEY),
]
for label, path, key in checks:
    r = requests.get(BASE + path, headers={"X-API-Key": key} if key else {}, timeout=10)
    print(f"{label} -> {r.status_code} {r.text}")

r = requests.post(BASE + "/predict/image", timeout=10,
                  files={"file": ("digit.png", b"", "image/png")})
print(f"키 없이 /predict/image -> {r.status_code} {r.text}")

2026-10-30 16:54:12 INFO     [image_api] MNIST 모델 로드 중...
2026-10-30 16:54:12 INFO     [image_api] 모델 로드 완료


GET /whoami → 401 (0.002s)
GET /whoami → 401 (0.002s)
POST /predict/image → 401 (0.002s)


헬스체크, 키 없음 -> 200 {"status":"healthy"}
키 없음 -> 401 {"detail":"API Key 가 필요합니다. X-API-Key 헤더를 넣어 주십시오"}
7.2절의 test-key-001 -> 401 {"detail":"등록되지 않은 API Key 입니다"}
내 키 -> 200 {"user":"대시보드"}
키 없이 /predict/image -> 401 {"detail":"API Key 가 필요합니다. X-API-Key 헤더를 넣어 주십시오"}


키가 없거나 목록에 없는 요청은 401로 끝났고 `/predict/image`도 같은 함수로 막혔습니다.

`/health`만 키 없이 열려 있습니다. 모니터링 도구가 서버가 살아 있는지 물을 수 있어야 하고 헬스체크의 답에는 서버 상태 한 줄만 있기 때문입니다. **어디를 열어 둘지도 설계입니다.**

In [14]:
# 12.3절 서버를 내립니다
hashed_server.should_exit = True
hashed_thread.join(timeout=10)
print("8490번 서버 :", "정지" if not hashed_thread.is_alive() else "아직 내려가지 않았습니다")

8490번 서버 : 정지


### 12.4 API Key가 답하지 못하는 것과 인증을 거는 자리

키 목록에 적힌 이름은 사람이 아니라 부르는 프로그램입니다. 대시보드를 여는 사람이 열 명이어도 대시보드는 키 하나로 API 서버를 부르므로 11장 로그의 `사용자:` 자리에는 열 명 모두 `대시보드`로 남습니다.

API Key로는 요청이 등록된 프로그램에서 왔는지와 어느 프로그램인지까지만 가립니다. 그 프로그램 뒤의 사람이 누구인지, 그 사람이 이 결과를 봐도 되는지는 답하지 못합니다. 사람 단위의 신원과 권한은 보안 주간의 개인정보와 접근 제어 노트에서 다룹니다.

그래도 키는 프로그램마다 따로 발급합니다. 키 하나가 새면 키 목록에서 그 줄만 지우고 새 키를 발급하면 되고 다른 프로그램은 멈추지 않습니다.

3.2절의 그림처럼 API 서버를 부르는 쪽은 대시보드 하나가 아닙니다. 대시보드에만 비밀번호를 걸면 모바일 앱이나 스크립트는 대시보드를 거치지 않고 API 서버를 바로 부릅니다. 11.3절의 테스트도 대시보드 없이 `requests`로 API 서버를 바로 불렀습니다.

**부르는 쪽이 여럿이므로 인증은 API 서버 한 곳에 겁니다.**

🔎 **더 알아보기**
API 서버가 여러 대로 늘면 인증을 그 앞, 모든 요청이 지나가는 공통 관문에 두기도 합니다. 그러면 서버가 늘어도 키 목록은 한 곳에 있고 `verify_api_key`는 그대로 쓰입니다. 그 관문은 게이트웨이 노트에서 다룹니다.

### 체크포인트

1. 키를 코드에 적으면 안 되는 이유를 두 가지 설명하십시오.  
2. 키 목록에 키 원문 대신 해시를 적으면 무엇이 달라집니까?  
3. 대시보드를 쓰는 사람이 열 명일 때 API Key로 알 수 없는 것은 무엇입니까?  
4. 대시보드에만 비밀번호를 걸면 안 되는 이유는 무엇입니까?

## 최종 체크포인트

```
Q1. API Key 인증이 없으면 어떤 위험이 있습니까? (두 가지 이상)
Q2. Depends(verify_api_key)는 엔드포인트 실행 전에 어떤 일을 합니까?
Q3. UploadFile 방식이 Base64보다 편리한 점은 무엇입니까?
Q4. 파일 업로드 시 크기 제한을 하지 않으면 어떤 문제가 생깁니까?
Q5. 이미지를 28x28 그레이스케일로 변환하는 이유는 무엇입니까?
```

---

### 요약

```
오늘 한 일:
  - API Key 인증의 필요성을 이해하고 구현했습니다.
  - Depends()를 사용하여 엔드포인트에 인증을 적용했습니다.
  - UploadFile로 이미지 파일 업로드를 구현했습니다.
  - 파일 크기/형식 검증, 이미지 리사이징 등 안전장치를 구현했습니다.
  - 인증 + 파일 업로드 + 모델 추론을 결합한 API를 완성했습니다.

다음 노트 (컨테이너 노트):
  - 지금까지 만든 API 서버와 대시보드를 내 노트북 말고 다른 기계에서도 똑같이 돌게 합니다.
  - 파이썬 판본과 꾸러미와 코드를 이미지 하나에 담고 API Key 같은 비밀값은 띄울 때 넣습니다.
```

### 제출

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다

1. 섹션 5 수행내역 캡쳐  
2. 섹션 11 수행내역 캡쳐, 11.8은 반드시 포함되어야 합니다  
3. 각 섹션 체크포인트의 답변

12.3절에서 키 해시 목록으로 401과 200이 갈린 출력도 함께 올리십시오.

수고하셨습니다!

## Reference

버전에 따른 동작은 2026년 10월 7일에 Streamlit 1.65.0, FastAPI 0.141.1, Pillow 12.3.0을 설치해 직접 확인했습니다.

**Streamlit**  
출처 : Streamlit, [st.testing.v1.AppTest](https://docs.streamlit.io/develop/api-reference/app-testing/st.testing.v1.apptest), 2026.09.21 확인  
출처 : Streamlit, [config.toml](https://docs.streamlit.io/develop/api-reference/configuration/config.toml), 2026.09.21 확인 (`server.address`, `browser.gatherUsageStats`, `server.maxUploadSize` 기본값은 1.65.0의 `streamlit config show`로 2026.10.07 확인)  
출처 : Streamlit, [st.button](https://docs.streamlit.io/develop/api-reference/widgets/st.button) (`use_container_width` 폐기 예정. 1.65.0 설치본의 설명문으로 2026.10.07 확인)

**FastAPI와 파이썬**  
출처 : FastAPI, [Lifespan Events](https://fastapi.tiangolo.com/advanced/events/) (`on_event` 폐기 예정. 0.141.1이 내는 경고 문구에 적힌 주소, 2026.10.07 확인)  
출처 : Python, [hmac.compare_digest](https://docs.python.org/3/library/hmac.html#hmac.compare_digest), 2026.09.21 확인

**Pillow와 코랩**  
출처 : Pillow, [Image 모듈](https://pillow.readthedocs.io/en/stable/reference/Image.html) (`Image.open`의 지연 로딩과 `MAX_IMAGE_PIXELS`. 값은 12.3.0 설치본으로 2026.10.07 확인)  
출처 : Google Colab, [colabtools `output/_util.py`](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py) (`serve_kernel_port_as_iframe`), 2026.10.07 확인  
출처 : Google Colab, [colabtools `userdata.py`](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py) (`userdata.get`), 2026.10.07 확인